# Supplier Cost Computation - Multiple Suppliers With REC Scenario

This notebook implements a comprehensive cost computation system for multiple suppliers in an energy market scenario with Renewable Energy Communities (REC). The analysis follows the energy market workflow from day-ahead forecasting to final customer billing.

## Overview
- **Scenario**: Multiple suppliers serving consumers and prosumers with REC structure
- **Market Sequence**: Day-ahead → Intra-day → Energy Community Settlement → Balancing Market → Supplier Billing
- **Data Source**: Consistent data from energy market operations in Austria (2016 data)
- **Suppliers**: Multiple suppliers with separate balancing groups

In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print(" Libraries imported successfully")
print(" Ready for supplier cost computation analysis")

 Libraries imported successfully
 Ready for supplier cost computation analysis


## Configuration and Data Loading

### Data Structure Documentation

Before loading the configuration, let's understand the data structure used in this analysis:

#### **Load Profiles (load_actual.csv, load_forecast_da.csv, load_forecast_id.csv)**

The load data contains **135 different load units** from the SimBench network `1-LV-urban6--2-no_sw`:

**Residential Loads:**
- `LV6.201 Load X [H0-A]` - Household type A profiles
- `LV6.201 Load X [H0-B]` - Household type B profiles
- `LV6.201 Load X [H0-C]` - Household type C profiles
- `LV6.201 Load X [H0-G]` - Household type G profiles
- `LV6.201 Load X [H0-L]` - Household type L profiles

**Commercial Loads:**
- `LV6.201 Load X [G1-A]`, `[G1-B]`, `[G1-C]` - Commercial type 1
- `LV6.201 Load X [G4-A]`, `[G4-B]` - Commercial type 4
- `LV6.201 Load X [G6-A]` - Commercial type 6

**Specialized Loads:**
- `LV6.201 Load X [HLS_A_3.7]`, `[HLS_B_3.7]`, `[HLS_C_3.7]` - Household Load Standard 3.7 kW
- `LV6.201 Load X [HLS_A_11.0]`, `[HLS_B_11.0]` - Household Load Standard 11.0 kW
- `LV6.201 Load X [APLS_A_50.0]`, `[APLS_B_11.0]` - Advanced Power Load Standard
- Heat pump profiles: `[Air_Alternative_2]`, `[Air_Parallel_2]`, `[Air_Semi-Parallel_2]`, `[Soil_Alternative_2]`

All loads are located on **Bus 201** of the LV6 network.

---

#### **Renewable Energy Sources (res_actual.csv, res_forecast_da.csv, res_forecast_id.csv)**

The RES data contains **12 solar PV units** from the SimBench network:

**Solar Photovoltaic (PV):**
- `LV6.201 SGen 1 [PV8]` - PV system type 8
- `LV6.201 SGen 2 [PV2]` - PV system type 2
- `LV6.201 SGen 3 [PV6]` - PV system type 6
- `LV6.201 SGen 4 [PV5]` - PV system type 5
- `LV6.201 SGen 5 [PV8]` - PV system type 8
- `LV6.201 SGen 6 [PV5]` - PV system type 5
- `LV6.201 SGen 7 [PV8]` - PV system type 8
- `LV6.201 SGen 8 [PV2]` - PV system type 2
- `LV6.201 SGen 9 [PV8]` - PV system type 8
- `LV6.201 SGen 10 [PV8]` - PV system type 8
- `LV6.201 SGen 11 [PV8]` - PV system type 8
- `LV6.201 SGen 12 [PV8]` - PV system type 8

All RES units are located on **Bus 201** of the LV6 network.

---

#### **Storage Systems (storage_actual.csv, storage_forecast_da.csv, storage_forecast_id.csv)**

The storage data contains **7 battery systems** paired with PV installations:

- `LV6.201 Storage 1 [Storage_PV5_H0-B]` - Battery paired with PV5 at household H0-B
- `LV6.201 Storage 2 [Storage_PV2_H0-A]` - Battery paired with PV2 at household H0-A
- `LV6.201 Storage 3 [Storage_PV8_H0-B]` - Battery paired with PV8 at household H0-B
- `LV6.201 Storage 4 [Storage_PV8_H0-L]` - Battery paired with PV8 at household H0-L
- `LV6.201 Storage 5 [Storage_PV8_H0-L]` - Battery paired with PV8 at household H0-L
- `LV6.201 Storage 6 [Storage_PV6_H0-B]` - Battery paired with PV6 at household H0-B
- `LV6.201 Storage 7 [Storage_PV8_H0-C]` - Battery paired with PV8 at household H0-C

All storage units are located on **Bus 201** of the LV6 network.

---

#### **Member Types in Configuration:**

**Consumers:**
- Contain **only load profiles**
- No generation capacity
- Pure energy buyers from suppliers
- Examples: households, commercial buildings

**Prosumers:**
- Contain **RES profiles** (renewable energy generation - PV)
- Can optionally have **load profiles** (energy consumption)
- Can optionally have **storage systems** (battery storage)
- Both consume and produce energy
- Examples: households with rooftop solar, commercial buildings with PV+storage

---

### Load JSON Configuration Templates
First, we load the JSON configurations that define the energy network structure, including suppliers, consumers, prosumers, and their relationships.


In [7]:
# Load B2 Multiple Suppliers With REC configuration
config_file = 'B2_multiple_supplier_with_rec_forecasts.json'

try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    print(f"Loaded configuration: {config_file}\n")
    
    # Display configuration summary
    print("="*80)
    print(" " * 25 + "SCENARIO CONFIGURATION")
    print("="*80)
    
    # System Information
    energy_system = config['energy_system']
    print(f"\nENERGY SYSTEM:")
    print(f"   - System ID: {energy_system['system_id']}")
    print(f"   - Name: {energy_system['system_name']}")
    print(f"   - Description: {energy_system['description']}")
    print(f"   - Location: {energy_system['location']['region']}, {energy_system['location']['country']}")
    print(f"   - Period: {energy_system['simulation_period']['start_date']} to {energy_system['simulation_period']['end_date']}")
    print(f"   - Timestep: {energy_system['simulation_period']['timestep']}")
    
    # Market Information
    energy_market = config['energy_market']
    print(f"\nENERGY MARKET:")
    print(f"   - Market ID: {energy_market['market_id']}")
    print(f"   - Market Name: {energy_market['market_name']}")
    print(f"   - Market Type: {energy_market['market_type']}")
    price_lists = energy_market['price_lists']
    print(f"   - Day-Ahead Prices: {price_lists['day_ahead_prices']['csv_file']}")
    print(f"   - Intraday Prices: {price_lists['intraday_prices']['csv_file']}")
    print(f"   - Grid Fees: {price_lists['grid_fees']['total_grid_fee_eur_per_kwh']} EUR/kWh")
    
    # Suppliers
    suppliers = config['suppliers']
    print(f"\nSUPPLIERS:")
    print(f"   - Total Suppliers: {len(suppliers)}")
    for supplier in suppliers:
        print(f"   • {supplier['supplier_id']}: {supplier['supplier_name']}")
        print(f"     - Balancing Group: {supplier['balancing_groups'][0]['balancing_group_id']}")
        print(f"     - Retail Pricing: {supplier['retail_pricing']['csv_file']} ({supplier['retail_pricing']['price']})")
        print(f"     - Feed-in Pricing: {supplier['feedin_pricing']['csv_file']} ({supplier['feedin_pricing']['price']})")
    
    # Prosumers and Consumers Distribution
    prosumers = config['prosumers']
    consumers = config['consumers']
    print(f"\nCUSTOMERS:")
    print(f"   - Total Prosumers: {len(prosumers)}")
    print(f"   - Total Consumers: {len(consumers)}")
    
    # Distribution by supplier
    for supplier in suppliers:
        supplier_id = supplier['supplier_id']
        prosumer_count = sum(1 for p in prosumers if p['supplier']['supplier_id'] == supplier_id)
        consumer_count = sum(1 for c in consumers if c['supplier']['supplier_id'] == supplier_id)
        print(f"   • {supplier_id}: {prosumer_count} prosumers + {consumer_count} consumers")
    
    # RECs
    recs = config['recs']
    print(f"\nRENEWABLE ENERGY COMMUNITIES:")
    print(f"   - Total RECs: {len(recs)}")
    if len(recs) == 0:
        print(f"   NO REC SCENARIO - Members trade individually with suppliers")
    else:
        for rec in recs:
            print(f"   • {rec['rec_id']}: {rec['rec_name']}")
            print(f"     - Type: {rec['rec_type']}")
            print(f"     - Description: {rec['description']}")
            print(f"     - Supplier: {rec['supplier']['supplier_id']} / {rec['supplier']['balancing_group_id']}")
    
    # Settlement Approach
    settlement = config['settlement_approach']
    print(f"\nSETTLEMENT APPROACH:")
    print(f"   - Type: {settlement['type'].upper()}")
    print(f"   - Description: {settlement['description']}")
    print(f"   - Internal Sharing: {settlement['internal_sharing']}")
    print(f"   - REC Bonuses: {settlement['rec_bonuses']}")
    print(f"   - Balancing Responsibility: {settlement['balancing_responsibility']}")
    print(f"   - Settlement Basis: {settlement['metering']['settlement_basis']}")
    
    print("\n" + "="*80)
    print("Configuration loaded successfully")
    print("="*80 + "\n")
    
except FileNotFoundError:
    print(f"File not found: {config_file}")
    print("Please ensure the configuration file exists in the B_Scenarion_Forecasting directory.")
except json.JSONDecodeError as e:
    print(f"Invalid JSON in: {config_file}")
    print(f"Error: {str(e)}")
except Exception as e:
    print(f"Error loading {config_file}: {str(e)}")

Loaded configuration: B2_multiple_supplier_with_rec_forecasts.json

                         SCENARIO CONFIGURATION

ENERGY SYSTEM:
   - System ID: ES_B2_MULTI_REC_001
   - Name: Multiple Suppliers with REC - Scenario B2
   - Description: 2 Suppliers with 2 RECs (one for consumers, one for prosumers). RECs perform internal energy sharing before external supplier billing. Settlement order: DA market â†’ ID market â†’ REC settlement â†’ Balancing market â†’ Supplier billing.
   - Location: Vienna, AT
   - Period: 2016-01-01 to 2016-12-31
   - Timestep: 15min

ENERGY MARKET:
   - Market ID: MARKET_001
   - Market Name: Austrian Energy Market
   - Market Type: day_ahead
   - Day-Ahead Prices: data/prices.csv
   - Intraday Prices: data/prices.csv
   - Grid Fees: 0.02 EUR/kWh

SUPPLIERS:
   - Total Suppliers: 2
   • SUP_A: Energy Supplier A
     - Balancing Group: BG_SUP_A_001
     - Retail Pricing: data/prices.csv (retail_price)
     - Feed-in Pricing: data/prices.csv (feedin_price)
   • SU

## Data Loading

Load the energy system data including prices, loads, RES forecasts, and actual values.

**Price Data (prices.csv):**
```
datetime,DA_price,ID_price,imbalance_price,retail_price,feedin_price
2016-01-01 00:00:00,27.69,27.98,-0.02,201.0,82.4
2016-01-01 00:15:00,27.69,27.98,-0.02,201.0,82.4
...
```

Contains market prices (EUR/MWh):
- `DA_price` - Day-ahead market prices (EUR/MWh)
- `ID_price` - Intraday market prices (EUR/MWh)
- `imbalance_price` - Single imbalance price for balancing market (EUR/MWh)
  - Negative: System has excess generation (surplus) → generators penalized, consumers rewarded
  - Positive: System has deficit (shortage) → generators rewarded, consumers penalized
- `retail_price` - Fixed retail electricity price for consumer purchases (EUR/MWh)
- `feedin_price` - Fixed feed-in tariff for prosumer exports to grid (EUR/MWh)

**Note on Feed-in Price:**
The feed-in price is **FIXED at €82.40/MWh** throughout the entire simulation period. This represents a constant feed-in tariff that suppliers pay to prosumers for energy exported to the grid. Unlike the day-ahead market price which varies with market conditions, the feed-in price remains stable to provide predictable compensation for distributed renewable energy generation.

In [8]:
# Load energy system data from configuration
# Notebook is in B_Scenarion_Forecasting/, data is in ../data/
# JSON paths start with 'data/', so we need to go up one level first

print("📊 LOADING ENERGY SYSTEM DATA FROM CONFIGURATION")
print("="*80)

# Load prices (common to all)
prices_filepath = Path('..') / config['energy_market']['price_lists']['day_ahead_prices']['csv_file']
prices_df = pd.read_csv(prices_filepath)
prices_df['datetime'] = pd.to_datetime(prices_df['datetime'])
prices_df.set_index('datetime', inplace=True)
print(f"✓ Loaded prices: {prices_df.shape}")
print(f"  Columns: {list(prices_df.columns)}")

# Load actual consumption/generation data
load_actual_filepath = Path('..') / config['settlement_approach']['metering']['load_actual']
load_actual_df = pd.read_csv(load_actual_filepath)
load_actual_df['datetime'] = pd.to_datetime(load_actual_df['datetime'])
load_actual_df.set_index('datetime', inplace=True)
print(f"✓ Loaded load_actual: {load_actual_df.shape}")

res_actual_filepath = Path('..') / config['settlement_approach']['metering']['res_actual']
res_actual_df = pd.read_csv(res_actual_filepath)
res_actual_df['datetime'] = pd.to_datetime(res_actual_df['datetime'])
res_actual_df.set_index('datetime', inplace=True)
print(f"✓ Loaded res_actual: {res_actual_df.shape}")

# Build forecast DataFrames from individual consumer/prosumer configurations
print("\n📊 Building forecast datasets from member configurations...")

# Initialize empty DataFrames
load_forecast_da_df = pd.DataFrame()
load_forecast_id_df = pd.DataFrame()
res_forecast_da_df = pd.DataFrame()
res_forecast_id_df = pd.DataFrame()

# Load consumer forecasts
print(f"\nLoading {len(config['consumers'])} consumer forecasts:")
for consumer in config['consumers']:
    member_id = consumer['load']['id']
    
    # Day-Ahead load forecast
    da_file = Path('..') / consumer['load']['load_forecast_da_file']
    da_df = pd.read_csv(da_file)
    da_df['datetime'] = pd.to_datetime(da_df['datetime'])
    da_df.set_index('datetime', inplace=True)
    if member_id in da_df.columns:
        load_forecast_da_df[member_id] = da_df[member_id]
    
    # Intra-Day load forecast
    id_file = Path('..') / consumer['load']['load_forecast_id_file']
    id_df = pd.read_csv(id_file)
    id_df['datetime'] = pd.to_datetime(id_df['datetime'])
    id_df.set_index('datetime', inplace=True)
    if member_id in id_df.columns:
        load_forecast_id_df[member_id] = id_df[member_id]
    
    print(f"  ✓ {consumer['meter_id']}: {member_id}")

# Load prosumer forecasts
print(f"\nLoading {len(config['prosumers'])} prosumer forecasts:")
for prosumer in config['prosumers']:
    member_id = prosumer['res']['id']
    
    # Day-Ahead RES forecast
    da_file = Path('..') / prosumer['res']['rec_res_forecast_da_file']
    da_df = pd.read_csv(da_file)
    da_df['datetime'] = pd.to_datetime(da_df['datetime'])
    da_df.set_index('datetime', inplace=True)
    if member_id in da_df.columns:
        res_forecast_da_df[member_id] = da_df[member_id]
    
    # Intra-Day RES forecast
    id_file = Path('..') / prosumer['res']['rec_res_forecast_id_file']
    id_df = pd.read_csv(id_file)
    id_df['datetime'] = pd.to_datetime(id_df['datetime'])
    id_df.set_index('datetime', inplace=True)
    if member_id in id_df.columns:
        res_forecast_id_df[member_id] = id_df[member_id]
    
    print(f"  ✓ {prosumer['meter_id']}: {member_id}")

# Store in es_data dictionary for compatibility
es_data = {
    'prices': prices_df,
    'load_actual': load_actual_df,
    'res_actual': res_actual_df,
    'load_forecast_da': load_forecast_da_df,
    'res_forecast_da': res_forecast_da_df,
    'load_forecast_id': load_forecast_id_df,
    'res_forecast_id': res_forecast_id_df
}

print(f"\n✓ Total datasets loaded: {len(es_data)}")
print(f"\n📈 Prices: {prices_df.index.min()} to {prices_df.index.max()}")
print(f"   - Avg DA price: €{prices_df['DA_price'].mean():.2f}/MWh")
print(f"   - Avg ID price: €{prices_df['ID_price'].mean():.2f}/MWh")
print(f"\n📊 Load Forecast DA: {load_forecast_da_df.shape} ({len(load_forecast_da_df.columns)} consumers)")
print(f"⚡ RES Forecast DA: {res_forecast_da_df.shape} ({len(res_forecast_da_df.columns)} prosumers)")
print(f"📊 Load Forecast ID: {load_forecast_id_df.shape} ({len(load_forecast_id_df.columns)} consumers)")
print(f"⚡ RES Forecast ID: {res_forecast_id_df.shape} ({len(res_forecast_id_df.columns)} prosumers)")
print("="*80)


📊 LOADING ENERGY SYSTEM DATA FROM CONFIGURATION
✓ Loaded prices: (35136, 5)
  Columns: ['DA_price', 'ID_price', 'imbalance_price', 'retail_price', 'feedin_price']
✓ Loaded load_actual: (35136, 118)
✓ Loaded res_actual: (35136, 17)

📊 Building forecast datasets from member configurations...

Loading 9 consumer forecasts:
  ✓ consumer_001: LV3.101 Load 1 [H0-A]
  ✓ consumer_002: LV3.101 Load 42 [H0-G]
  ✓ consumer_003: LV3.101 Load 24 [H0-A]
  ✓ consumer_004: LV3.101 Load 11 [H0-A]
  ✓ consumer_005: LV3.101 Load 20 [H0-A]
  ✓ consumer_006: LV3.101 Load 27 [H0-A]
  ✓ consumer_007: LV3.101 Load 30 [H0-A]
  ✓ consumer_008: LV3.101 Load 36 [H0-A]
  ✓ consumer_009: LV3.101 Load 37 [H0-A]

Loading 6 prosumer forecasts:
  ✓ prosumer_001: LV3.101 SGen 1 [PV8]
  ✓ prosumer_002: LV3.101 SGen 2 [PV2]
  ✓ prosumer_003: LV3.101 SGen 3 [PV6]
  ✓ prosumer_004: LV3.101 SGen 4 [PV4]
  ✓ prosumer_005: LV3.101 SGen 2 [PV3]
  ✓ prosumer_006: LV3.101 SGen 6 [PV5]

✓ Total datasets loaded: 7

📈 Prices: 2016-0

## 1. Day-Ahead Market Operations

### 1.1 Day-Ahead Market Participation
Calculate day-ahead market positions for each balancing group based on aggregated forecasts of all members (consumers and prosumers).

**Aggregation Process:**
- Each balancing group aggregates (sums) load and RES forecasts from all assigned members
- Aggregated forecasts are submitted to the day-ahead market before gate closure
- Market positions are determined at the balancing group level, not individually

**Day-Ahead Market Commitments:**
- **DA Load Forecast (MWh)**: Σ(member load forecasts) - Total consumption (always ≥ 0)
- **DA Gen Forecast (MWh)**: Σ(member generation forecasts) - Total generation (always ≥ 0)
- **DA Purchase Commitment (EUR)**: Load × DA_price - Commitment for purchases (always ≥ 0)
- **DA Sale Commitment (EUR)**: Gen × DA_price - Commitment from sales (always ≥ 0)

In [9]:
def calculate_da_market_commitments(load_da_df, gen_da_df, prices_df, config):
    """
    Calculate day-ahead market commitments and create energy system DataFrame
    
    Process:
    1. Aggregate DA forecasts by supplier and balancing group
    2. Calculate net position (gen - load)
    3. Split net position into separate purchase/sale commitments
    4. Return DataFrame with time-series data
    
    Net Position Logic:
    - Net position = sum_gen - sum_load
    - If net < 0 (deficit): need to purchase |net| from market
      * da_net_load_forecast_mwh = |net| (amount to purchase)
      * da_net_gen_forecast_mwh = 0
    - If net > 0 (surplus): can sell net to market
      * da_net_load_forecast_mwh = 0
      * da_net_gen_forecast_mwh = net (amount to sell)
    - If net = 0 (balanced): no market transaction needed
    
    Separate Purchase/Sale Accounting:
    - da_net_load_forecast_mwh: Net load to purchase from market (always ≥ 0)
    - da_net_gen_forecast_mwh: Net generation to sell to market (always ≥ 0)
    - da_purchase_commitment_eur: Cost for buying energy (always ≥ 0)
    - da_sale_commitment_eur: Revenue from selling energy (always ≥ 0)
    
    Returns DataFrame with columns: datetime, supplier_id, balancing_group_id,
    da_net_load_forecast_mwh, da_net_gen_forecast_mwh, da_price_eur_per_mwh,
    da_purchase_commitment_eur, da_sale_commitment_eur
    """
    # Step 1: Create aggregated forecast DataFrame
    da_forecast_df = pd.DataFrame(index=load_da_df.index)
    
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Initialize aggregated forecasts
            bg_load_total = pd.Series(0, index=da_forecast_df.index)
            bg_gen_total = pd.Series(0, index=da_forecast_df.index)
            
            # Aggregate prosumers in this balancing group
            for prosumer in config['prosumers']:
                if prosumer['supplier']['supplier_id'] == supplier_id and \
                   prosumer['supplier']['balancing_group_id'] == bg_id:
                    
                    # Add load forecast if exists
                    if 'load' in prosumer and prosumer['load']:
                        load_id = prosumer['load']['id']
                        if load_id in load_da_df.columns:
                            bg_load_total += load_da_df[load_id]
                    
                    # Add gen forecast if exists
                    if 'res' in prosumer and prosumer['res']:
                        gen_id = prosumer['res']['id']
                        if gen_id in gen_da_df.columns:
                            bg_gen_total += gen_da_df[gen_id]
            
            # Aggregate consumers in this balancing group
            for consumer in config['consumers']:
                if consumer['supplier']['supplier_id'] == supplier_id and \
                   consumer['supplier']['balancing_group_id'] == bg_id:
                    
                    load_id = consumer['load']['id']
                    if load_id in load_da_df.columns:
                        bg_load_total += load_da_df[load_id]
            
            # Calculate net position (gen - load)
            net_position = bg_gen_total - bg_load_total
            
            # Split into purchase/sale based on net position sign
            # If deficit (net < 0): purchase = |net|, sale = 0
            # If surplus (net > 0): purchase = 0, sale = net
            bg_load_forecast = net_position.clip(upper=0).abs()  # max(0, -net) = purchase amount
            bg_gen_forecast = net_position.clip(lower=0)         # max(0, net) = sale amount
            
            # Add to DataFrame
            da_forecast_df[f"{bg_id}_load"] = bg_load_forecast
            da_forecast_df[f"{bg_id}_gen"] = bg_gen_forecast
    
    # Step 2: Calculate DA market transactions
    da_data_list = []
    
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Get forecasts for this balancing group
            da_load_forecast = da_forecast_df[f"{bg_id}_load"]
            da_gen_forecast = da_forecast_df[f"{bg_id}_gen"]
            
            # Get DA price
            da_price = prices_df['DA_price']
            
            # Calculate purchase cost and sale revenue separately
            # Purchase cost = load purchased from market
            # Sale revenue = generation sold to market
            purchase_cost_eur = da_load_forecast * da_price
            sale_revenue_eur = da_gen_forecast * da_price
            
            # Store time series data
            for timestamp in da_forecast_df.index:
                da_data_list.append({
                    'datetime': timestamp,
                    'supplier_id': supplier_id,
                    'balancing_group_id': bg_id,
                    'da_net_load_forecast_mwh': da_load_forecast.loc[timestamp],
                    'da_net_gen_forecast_mwh': da_gen_forecast.loc[timestamp],
                    'da_price_eur_per_mwh': da_price.loc[timestamp],
                    'da_purchase_commitment_eur': purchase_cost_eur.loc[timestamp],
                    'da_sale_commitment_eur': sale_revenue_eur.loc[timestamp]
                })
    
    # Create DataFrame
    es_timeseries_df = pd.DataFrame(da_data_list)
    
    return es_timeseries_df

# Calculate day-ahead market commitments and update es_timeseries_df
es_timeseries_df = calculate_da_market_commitments(
    es_data['load_forecast_da'],
    es_data['res_forecast_da'],
    es_data['prices'],
    config
)

print("Sample of es_timeseries_df after DA market:")
display(es_timeseries_df.head())

Sample of es_timeseries_df after DA market:


""


### 1.2 Day-Ahead Market - Mathematical Formulation

This section documents the mathematical formulation of the **day-ahead market commitment mechanism** implemented in the `calculate_da_market_commitments()` function.

---

#### **Column Definitions and Mathematical Formulas**

The day-ahead market calculation creates the following columns in `es_timeseries_df`:

##### **1. Day-Ahead Net Forecast** (`da_net_forecast_mwh`)

$$
Q_{\text{DA,forecast}}^{t} = Q_{\text{RES,DA}}^{t} - Q_{\text{load,DA}}^{t}
$$

Where:
- $Q_{\text{DA,forecast}}^{t}$ = Day-ahead net forecast at timestamp $t$ (MWh)
- $Q_{\text{RES,DA}}^{t}$ = Day-ahead RES generation forecast at timestamp $t$ (MWh)
- $Q_{\text{load,DA}}^{t}$ = Day-ahead load consumption forecast at timestamp $t$ (MWh)

**Aggregation Formula:**

For each balancing group:

$$
Q_{\text{RES,DA}}^{t} = \sum_{i \in \text{prosumers}} Q_{\text{RES},i}^{t}
$$

$$
Q_{\text{load,DA}}^{t} = \sum_{i \in \text{prosumers}} Q_{\text{load},i}^{t} + \sum_{j \in \text{consumers}} Q_{\text{load},j}^{t}
$$

**Sign Convention:**
- **Positive** ($Q_{\text{DA,forecast}}^{t} > 0$): **SURPLUS** position
  * RES generation exceeds load consumption
  * BRP can **sell** energy to the market
  
- **Negative** ($Q_{\text{DA,forecast}}^{t} < 0$): **DEFICIT** position
  * Load consumption exceeds RES generation
  * BRP needs to **buy** energy from the market

**Physical Interpretation:**

| Net Forecast | Meaning | BRP Position | Market Action |
|--------------|---------|--------------|---------------|
| +50 MWh | 50 MWh surplus | Long (seller) | Sell 50 MWh |
| -50 MWh | 50 MWh deficit | Short (buyer) | Buy 50 MWh |
| 0 MWh | Balanced | Flat | No trade needed |

---

##### **2. Day-Ahead Price** (`da_price_eur_per_mwh`)

$$
p_{\text{DA}}^{t} = \text{DA\_price}^{t} \quad \text{(EUR/MWh)}
$$

Where:
- $p_{\text{DA}}^{t}$ = Day-ahead market clearing price at timestamp $t$ (EUR/MWh)

**Market Context:** 
- Day-ahead prices determined by **market coupling** across European power exchanges
- **Auction mechanism**: Single clearing price for each hour/quarter-hour
- Cleared one day before delivery (typically at 12:00 noon for next day)
- Reflects supply/demand equilibrium based on D-1 forecasts

**Price Characteristics:**
- **Typical range**: 20-100 EUR/MWh (normal conditions)
- **Negative prices**: Possible during high renewable generation + low demand
- **Price spikes**: Can exceed 500 EUR/MWh during scarcity events
- **Seasonal patterns**: Higher in winter (heating demand), lower in summer

---

##### **3. Day-Ahead Net Commitment** (`da_net_commitment_eur`)

$$
S_{\text{DA}}^{t} = Q_{\text{DA,forecast}}^{t} \times p_{\text{DA}}^{t}
$$

Where:
- $S_{\text{DA}}^{t}$ = Day-ahead net commitment at timestamp $t$ (EUR)

**Economic Sign Convention:**
- **Positive** ($S_{\text{DA}}^{t} > 0$): **NET REVENUE** (money in)
  * BRP sells more than buys
  * Revenue from selling surplus energy
  
- **Negative** ($S_{\text{DA}}^{t} < 0$): **NET COST** (money out)
  * BRP buys more than sells
  * Cost for purchasing deficit energy

**Settlement Examples:**

| Forecast | DA Price | Commitment | Interpretation |
|----------|----------|------------|----------------|
| +50 MWh (surplus) | 60 EUR/MWh | **+3,000 EUR** | BRP sells 50 MWh → receives 3,000 EUR |
| -50 MWh (deficit) | 60 EUR/MWh | **-3,000 EUR** | BRP buys 50 MWh → pays 3,000 EUR |
| +50 MWh (surplus) | -10 EUR/MWh | **-500 EUR** | BRP pays to dispose surplus (negative price) |
| -100 MWh (deficit) | 200 EUR/MWh | **-20,000 EUR** | BRP buys at high price → high cost |

---

#### **Total Day-Ahead Market Cost/Revenue**

The total day-ahead commitment for a BRP over the entire period:

$$
C_{\text{DA,total}} = \sum_{t=1}^{T} S_{\text{DA}}^{t} = \sum_{t=1}^{T} \left[ Q_{\text{DA,forecast}}^{t} \times p_{\text{DA}}^{t} \right]
$$

Where:
- $T$ = Total number of timestamps (35,136 for one year at 15-minute resolution)
- **Negative total**: Net buyer position (BRP pays for energy)
- **Positive total**: Net seller position (BRP receives payment)

**Decomposition by Buy/Sell:**

$$
C_{\text{DA,total}} = C_{\text{purchases}} + R_{\text{sales}}
$$

Where:
- $C_{\text{purchases}} = \sum_{t: Q_{\text{DA,forecast}}^{t} < 0} Q_{\text{DA,forecast}}^{t} \times p_{\text{DA}}^{t}$ (always negative)
- $R_{\text{sales}} = \sum_{t: Q_{\text{DA,forecast}}^{t} > 0} Q_{\text{DA,forecast}}^{t} \times p_{\text{DA}}^{t}$ (always positive)

---

#### **Market Participation Logic**

**For Surplus Position** ($Q_{\text{DA,forecast}}^{t} > 0$):
1. BRP submits **sell order** to day-ahead market
2. If $p_{\text{DA}}^{t} \geq$ BRP's minimum acceptable price → Order accepted
3. BRP commits to **deliver** $Q_{\text{DA,forecast}}^{t}$ MWh
4. BRP receives $S_{\text{DA}}^{t}$ EUR (positive revenue)

**For Deficit Position** ($Q_{\text{DA,forecast}}^{t} < 0$):
1. BRP submits **buy order** to day-ahead market
2. If $p_{\text{DA}}^{t} \leq$ BRP's maximum acceptable price → Order accepted
3. BRP commits to **receive** $|Q_{\text{DA,forecast}}^{t}|$ MWh
4. BRP pays $S_{\text{DA}}^{t}$ EUR (negative cost)

---

#### **Portfolio Aggregation**

Each balancing group aggregates multiple members:

**Total Load Forecast:**
$$
Q_{\text{load,DA}}^{t} = \sum_{k=1}^{N_{\text{prosumers}}} Q_{\text{load},k}^{t} + \sum_{m=1}^{N_{\text{consumers}}} Q_{\text{load},m}^{t}
$$

**Total RES Forecast:**
$$
Q_{\text{RES,DA}}^{t} = \sum_{k=1}^{N_{\text{prosumers}}} Q_{\text{RES},k}^{t}
$$

**Portfolio Net Position:**
$$
Q_{\text{DA,forecast}}^{t} = Q_{\text{RES,DA}}^{t} - Q_{\text{load,DA}}^{t}
$$

**Benefits of Portfolio Aggregation:**
- **Diversification**: Individual forecast errors partially cancel out
- **Reduced volatility**: Portfolio variance < sum of individual variances
- **Better hedging**: Mixed load/generation creates natural hedge
- **Risk reduction**: Correlation effects improve overall forecast accuracy

---

#### **Timing and Gate Closure**

**Day-Ahead Market Timeline (D-1 for delivery day D):**

```
D-1, 08:00-12:00  →  Market participants submit bids/offers
    ↓
D-1, 12:00        →  GATE CLOSURE (no more bids accepted)
    ↓
D-1, 12:30-13:00  →  Market clearing algorithm runs
    ↓
D-1, 13:00-14:00  →  Results published (clearing prices + volumes)
    ↓
D, 00:00-24:00    →  Physical delivery obligation begins
```

**Key Timing Rules:**
- Forecasts must be submitted **before 12:00 on D-1**
- Once cleared, positions are **financially binding**
- Physical delivery occurs **next day (D)**, typically in hourly or 15-min intervals
- No changes allowed after gate closure (must use intra-day market for adjustments)

---

#### **Data Structure Notes**

1. **Balancing Group Level**: 
   - Each supplier operates one or more balancing groups
   - All calculations performed at BG level
   - Members assigned to specific BG within their supplier

2. **Temporal Resolution**: 
   - **15-minute intervals** (Austrian/German market standard)
   - Total periods per year: **35,136** (4 × 24 × 366)
   - Market-to-market (MTM) settlement at each interval

3. **Forecast Horizon**:
   - DA forecasts made **12-36 hours** before delivery
   - Weather uncertainty still significant
   - Load patterns more predictable than renewables

---

#### **Risk Considerations**

**Price Risk:**
- DA prices set financial exposure for next day
- Price volatility risk: unexpected price spikes/drops
- Hedging strategy: Balance spot exposure with forward contracts

**Volume Risk:**
- Forecast errors create imbalance exposure
- Larger portfolios → better forecast accuracy (portfolio effect)
- Typical forecast error: 10-30% for RES, 2-5% for load

**Market Power:**
- Large players may influence prices
- Regulatory oversight ensures competitive bidding
- Price caps/floors prevent extreme manipulation

---

#### **Summary Statistics Interpretation**

**Net Position Ratio:**
$$
R_{\text{net}} = \frac{\sum_{t=1}^{T} Q_{\text{DA,forecast}}^{t}}{\sum_{t=1}^{T} Q_{\text{load,DA}}^{t}} \times 100\%
$$

- **Positive ratio**: Net seller (more RES than load)
- **Negative ratio**: Net buyer (more load than RES)
- **Close to zero**: Balanced portfolio

**Average Settlement Price:**
$$
\bar{p}_{\text{effective}} = \frac{C_{\text{DA,total}}}{\sum_{t=1}^{T} |Q_{\text{DA,forecast}}^{t}|}
$$

Represents the **effective price** paid/received per MWh traded (accounting for buy/sell mix).

---

#### **Connection to Subsequent Markets**

The day-ahead commitment serves as the **baseline** for all subsequent market operations:

1. **Intra-Day Market**: Adjust DA position based on updated forecasts
2. **Balancing Market**: Settle deviations from final committed position (DA + ID)
3. **Retail Billing**: Supplier's wholesale costs form basis for retail pricing

**Financial Flow:**
$$
\text{Total Cost} = C_{\text{DA}} + C_{\text{ID}} + C_{\text{Balancing}} + C_{\text{Other}}
$$

The day-ahead commitment ($C_{\text{DA}}$) typically represents **70-90%** of total wholesale costs for most suppliers.

## 2. Intra-Day Market Operations

Calculate intra-day forecasts and adjustments from day-ahead commitments.

In [10]:
def calculate_id_market_adjustments(es_timeseries_df, load_forecast_id, gen_forecast_id, prices_df, config):
    """
    Calculate intra-day market adjustments and add to energy system DataFrame
    
    Process:
    1. Aggregate ID forecasts by supplier and balancing group (total load and gen)
    2. Calculate ID net position and split into net load/gen (same as DA logic)
    3. Calculate adjustments from DA net commitments (extracted from es_timeseries_df)
    4. Add ID adjustment columns to es_timeseries_df
    
    ID Net Position Logic (builds from total forecasts):
    - Calculate: id_net = id_gen_total - id_load_total
    - Split: id_net_load = max(-id_net, 0), id_net_gen = max(id_net, 0)
    - Adjustment: change from DA net position (ID - DA)
    
    Columns Created:
    - id_net_load_forecast_mwh: Net load to purchase in ID market (always ≥ 0)
    - id_net_gen_forecast_mwh: Net generation to sell in ID market (always ≥ 0)
    - id_net_load_adjustment_mwh: Change in net load from DA (can be + or -)
    - id_net_gen_adjustment_mwh: Change in net gen from DA (can be + or -)
    - id_purchase_adjustment_eur: Additional purchase cost/savings (can be + or -)
    - id_sale_adjustment_eur: Additional sale revenue/reduction (can be + or -)
    - closing_net_load_forecast_mwh: Total net load position after ID (DA + ID adjustment)
    - closing_net_gen_forecast_mwh: Total net gen position after ID (DA + ID adjustment)
    
    Returns updated es_timeseries_df with ID adjustment columns added
    """
    # Step 1: Create ID forecast DataFrame by aggregating member forecasts
    id_forecast_df = pd.DataFrame(index=load_forecast_id.index)
    
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Initialize aggregated forecasts
            bg_load_forecast = pd.Series(0, index=id_forecast_df.index)
            bg_gen_forecast = pd.Series(0, index=id_forecast_df.index)
            
            # Aggregate prosumers in this balancing group
            for prosumer in config['prosumers']:
                if prosumer['supplier']['supplier_id'] == supplier_id and \
                   prosumer['supplier']['balancing_group_id'] == bg_id:
                    
                    # Add load forecast if exists
                    if 'load' in prosumer and prosumer['load']:
                        load_id = prosumer['load']['id']
                        if load_id in load_forecast_id.columns:
                            bg_load_forecast += load_forecast_id[load_id]
                    
                    # Add gen forecast if exists
                    if 'res' in prosumer and prosumer['res']:
                        gen_id = prosumer['res']['id']
                        if gen_id in gen_forecast_id.columns:
                            bg_gen_forecast += gen_forecast_id[gen_id]
            
            # Aggregate consumers in this balancing group
            for consumer in config['consumers']:
                if consumer['supplier']['supplier_id'] == supplier_id and \
                   consumer['supplier']['balancing_group_id'] == bg_id:
                    
                    load_id = consumer['load']['id']
                    if load_id in load_forecast_id.columns:
                        bg_load_forecast += load_forecast_id[load_id]
            
            # Add to DataFrame
            id_forecast_df[f"{bg_id}_load"] = bg_load_forecast
            id_forecast_df[f"{bg_id}_gen"] = bg_gen_forecast
    
    # Step 2: Calculate ID adjustments and add to es_timeseries_df
    id_data_list = []
    
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Get DA forecasts from es_timeseries_df
            da_data = es_timeseries_df[
                (es_timeseries_df['supplier_id'] == supplier_id) & 
                (es_timeseries_df['balancing_group_id'] == bg_id)
            ].set_index('datetime')
            
            da_load_forecast = da_data['da_net_load_forecast_mwh']
            da_gen_forecast = da_data['da_net_gen_forecast_mwh']
            
            # Get ID total forecasts for this balancing group
            id_load_total = id_forecast_df[f"{bg_id}_load"]
            id_gen_total = id_forecast_df[f"{bg_id}_gen"]
            
            # Calculate ID net position and split (same logic as DA)
            id_net_position = id_gen_total - id_load_total
            id_net_load_forecast = id_net_position.clip(upper=0).abs()  # max(0, -net)
            id_net_gen_forecast = id_net_position.clip(lower=0)         # max(0, net)
            
            # Calculate adjustments from DA net positions (ID - DA)
            load_adjustment = id_net_load_forecast - da_load_forecast
            gen_adjustment = id_net_gen_forecast - da_gen_forecast
            
            # Get ID prices
            id_price = prices_df['ID_price']
            
            # Calculate adjustment costs and revenues
            purchase_adjustment_eur = load_adjustment * id_price
            sale_adjustment_eur = gen_adjustment * id_price
            
            # Calculate closing net positions (DA + ID)
            closing_net_load = da_load_forecast + load_adjustment
            closing_net_gen = da_gen_forecast + gen_adjustment
            
            # Store time series data for merging
            for timestamp in id_forecast_df.index:
                id_data_list.append({
                    'datetime': timestamp,
                    'supplier_id': supplier_id,
                    'balancing_group_id': bg_id,
                    'id_net_load_forecast_mwh': id_net_load_forecast.loc[timestamp],
                    'id_net_gen_forecast_mwh': id_net_gen_forecast.loc[timestamp],
                    'id_net_load_adjustment_mwh': load_adjustment.loc[timestamp],
                    'id_net_gen_adjustment_mwh': gen_adjustment.loc[timestamp],
                    'id_price_eur_per_mwh': id_price.loc[timestamp],
                    'id_purchase_adjustment_eur': purchase_adjustment_eur.loc[timestamp],
                    'id_sale_adjustment_eur': sale_adjustment_eur.loc[timestamp],
                    'closing_net_load_forecast_mwh': closing_net_load.loc[timestamp],
                    'closing_net_gen_forecast_mwh': closing_net_gen.loc[timestamp]
                })
    
    # Create temporary DataFrame with ID data
    id_data_df = pd.DataFrame(id_data_list)
    
    # Drop any columns from id_data_df that already exist in es_timeseries_df
    # Keep only merge keys and new columns
    merge_keys = ['datetime', 'supplier_id', 'balancing_group_id']
    shared_columns = [col for col in id_data_df.columns 
                     if col in es_timeseries_df.columns and col not in merge_keys]
    
    if shared_columns:
        id_data_df = id_data_df.drop(columns=shared_columns)
    
    # Merge ID columns into es_timeseries_df
    es_timeseries_df = es_timeseries_df.merge(
        id_data_df,
        on=merge_keys,
        how='left'
    )
    
    # Calculate closing purchase and sale commitments (DA + ID)
    es_timeseries_df['closing_purchase_commitment_eur'] = (
        es_timeseries_df['da_purchase_commitment_eur'] + 
        es_timeseries_df['id_purchase_adjustment_eur']
    )
    es_timeseries_df['closing_sale_commitment_eur'] = (
        es_timeseries_df['da_sale_commitment_eur'] + 
        es_timeseries_df['id_sale_adjustment_eur']
    )
    
    return es_timeseries_df


# Calculate intra-day market adjustments and update es_timeseries_df
es_timeseries_df = calculate_id_market_adjustments(
    es_timeseries_df,
    es_data['load_forecast_id'],
    es_data['res_forecast_id'],
    es_data['prices'],
    config
)

# Display sample of time series data
display(es_timeseries_df.head())

KeyError: 'supplier_id'

### 2.1 Intra-Day Market - Mathematical Formulation

This section documents the mathematical formulation of the **intra-day market adjustment mechanism** implemented in the `calculate_id_market_adjustments()` function.

---

#### **Column Definitions and Mathematical Formulas**

The intra-day market calculation creates the following columns in `es_timeseries_df`:

##### **1. Intra-Day Net Forecast** (`id_net_forecast_mwh`)

$$
Q_{\text{ID,forecast}}^{t} = Q_{\text{RES,ID}}^{t} - Q_{\text{load,ID}}^{t}
$$

Where:
- $Q_{\text{ID,forecast}}^{t}$ = Intra-day net forecast at timestamp $t$ (MWh)
- $Q_{\text{RES,ID}}^{t}$ = Intra-day RES generation forecast at timestamp $t$ (MWh)
- $Q_{\text{load,ID}}^{t}$ = Intra-day load consumption forecast at timestamp $t$ (MWh)

**Sign Convention:**
- **Positive**: Forecasted surplus (generation > load)
- **Negative**: Forecasted deficit (generation < load)

**Purpose:** Updated forecast closer to real-time that refines the day-ahead forecast based on improved weather predictions, demand patterns, and operational conditions.

---

##### **2. Intra-Day Adjustment** (`id_adjustment_mwh`)

$$
\Delta Q_{\text{ID}}^{t} = Q_{\text{ID,forecast}}^{t} - Q_{\text{DA,forecast}}^{t}
$$

Where:
- $\Delta Q_{\text{ID}}^{t}$ = Intra-day adjustment at timestamp $t$ (MWh)
- $Q_{\text{DA,forecast}}^{t}$ = Day-ahead net forecast at timestamp $t$ (MWh)

**Sign Convention:**
- **Positive adjustment** ($\Delta Q_{\text{ID}}^{t} > 0$): 
  * ID forecast shows more surplus than DA forecast
  * BRP needs to **sell more** or **buy less** energy
  
- **Negative adjustment** ($\Delta Q_{\text{ID}}^{t} < 0$): 
  * ID forecast shows more deficit than DA forecast
  * BRP needs to **buy more** or **sell less** energy

**Physical Interpretation:**

| Scenario | DA Forecast | ID Forecast | Adjustment | BRP Action |
|----------|-------------|-------------|------------|------------|
| **1** | -100 MWh (deficit) | -80 MWh (deficit) | +20 MWh | Buy 20 MWh less |
| **2** | -100 MWh (deficit) | -120 MWh (deficit) | -20 MWh | Buy 20 MWh more |
| **3** | +100 MWh (surplus) | +120 MWh (surplus) | +20 MWh | Sell 20 MWh more |
| **4** | +100 MWh (surplus) | +80 MWh (surplus) | -20 MWh | Sell 20 MWh less |

---

##### **3. Intra-Day Price** (`id_price_eur_per_mwh`)

$$
p_{\text{ID}}^{t} = \text{DOD\_price}^{t} \quad \text{(EUR/MWh)}
$$

Where:
- $p_{\text{ID}}^{t}$ = Intra-day market price at timestamp $t$ (EUR/MWh)
- DOD = Day-of-Delivery market (Austrian/German intraday continuous trading)

**Market Context:** The intra-day price reflects real-time supply/demand balance and typically shows more volatility than day-ahead prices due to:
- Weather forecast updates
- Unplanned outages
- Demand variations
- Renewable generation deviations

---

##### **4. Intra-Day Net Adjustment Settlement** (`id_net_adjustment_eur`)

$$
S_{\text{ID}}^{t} = \Delta Q_{\text{ID}}^{t} \times p_{\text{ID}}^{t}
$$

Where:
- $S_{\text{ID}}^{t}$ = Intra-day net adjustment settlement at timestamp $t$ (EUR)

**Economic Sign Convention:**
- **Positive** ($S_{\text{ID}}^{t} > 0$): Net revenue (money in)
  * BRP sells more energy than DA commitment
  * Or buys less energy than DA commitment
  
- **Negative** ($S_{\text{ID}}^{t} < 0$): Net cost (money out)
  * BRP buys more energy than DA commitment
  * Or sells less energy than DA commitment

**Settlement Examples:**

| Adjustment | ID Price | Settlement | Interpretation |
|------------|----------|------------|----------------|
| +20 MWh | 50 EUR/MWh | **+1,000 EUR** | BRP sells 20 MWh → receives revenue |
| -20 MWh | 50 EUR/MWh | **-1,000 EUR** | BRP buys 20 MWh → pays cost |
| +20 MWh | -10 EUR/MWh | **-200 EUR** | BRP must pay to dispose surplus |
| -20 MWh | 100 EUR/MWh | **-2,000 EUR** | BRP buys at high price → high cost |

---

##### **5. Closing Net Commitment** (`closing_net_commitment_eur`)

$$
S_{\text{closing}}^{t} = S_{\text{DA}}^{t} + S_{\text{ID}}^{t}
$$

Where:
- $S_{\text{closing}}^{t}$ = Closing net commitment at timestamp $t$ (EUR)
- $S_{\text{DA}}^{t}$ = Day-ahead net commitment (EUR)
- $S_{\text{ID}}^{t}$ = Intra-day net adjustment (EUR)

**Purpose:** Represents the BRP's **total wholesale market commitment** before real-time balancing. This is the financial position the BRP has locked in through forward markets (DA + ID).

**Economic Interpretation:**
- **Negative closing commitment**: BRP expects to pay for net energy purchases
- **Positive closing commitment**: BRP expects to receive revenue from net energy sales

---

#### **Complete Intra-Day Market Formula**

The total intra-day adjustment cost/revenue for a BRP over the entire period:

$$
C_{\text{ID,total}} = \sum_{t=1}^{T} S_{\text{ID}}^{t} = \sum_{t=1}^{T} \left[ \left( Q_{\text{ID,forecast}}^{t} - Q_{\text{DA,forecast}}^{t} \right) \times p_{\text{ID}}^{t} \right]
$$

Where:
- $T$ = Total number of timestamps (e.g., 35,136 for one year at 15-minute resolution)
- **Negative total**: Net cost (BRP paid more in ID market)
- **Positive total**: Net revenue (BRP received more in ID market)

---

#### **Market Workflow Integration**

The intra-day market fits into the wholesale electricity market sequence:

```
Day-Ahead Market (D-1, 12:00)
    ↓
    Initial commitment based on D-1 forecast
    ↓
Intra-Day Market (D-1 to D, continuous)
    ↓
    Adjust position based on updated forecasts
    ↓
Gate Closure (varies by market, typically 5-60 min before delivery)
    ↓
    Closing position locked in
    ↓
Real-Time Delivery & Balancing Market
    ↓
    Settle actual deviations from closing position
```

---

#### **Data Aggregation Notes**

1. **Balancing Group Level**: All calculations performed at balancing group level
   - Each supplier has one or more balancing groups
   - Members (prosumers & consumers) are aggregated within their assigned balancing group

2. **Temporal Resolution**: 15-minute intervals
   - Matches standard European electricity market settlement periods
   - Total periods per year: 35,136 (4 intervals/hour × 24 hours/day × 366 days)

3. **Forecast Updates**: 
   - Day-Ahead (DA): Forecast made ~12-36 hours before delivery
   - Intra-Day (ID): Forecast made 1-24 hours before delivery (closer to real-time)
   - Expected improvement: ID forecasts typically 20-40% more accurate than DA

---

#### **Risk Management Considerations**

**Price Risk:**
- ID prices can be more volatile than DA prices
- Positive/negative price spikes possible during system stress
- BRPs must balance forecast accuracy vs. market price risk

**Volume Risk:**
- Forecast errors between DA and ID create adjustment needs
- Large adjustments may face liquidity constraints
- Optimal strategy balances forecast improvement vs. adjustment costs

**Arbitrage Opportunities:**
- Price spreads between DA and ID markets
- BRPs with good forecasting can profit from price differences
- However, transaction costs and forecast uncertainty limit arbitrage

---

#### **Summary Statistics Interpretation**

When analyzing ID market performance:

**Total ID Adjustment Volume:**
$$
V_{\text{ID}} = \sum_{t=1}^{T} |\Delta Q_{\text{ID}}^{t}|
$$

**Average Adjustment Rate:**
$$
R_{\text{adj}} = \frac{V_{\text{ID}}}{\sum_{t=1}^{T} |Q_{\text{DA,forecast}}^{t}|} \times 100\%
$$

**Forecast Improvement:**
- Lower adjustment rates indicate better ID forecasts
- Typical rates: 10-30% depending on renewable penetration and forecast quality

## 3. Energy Community Settlement

### 3.1 REC Internal Energy Sharing

In the B2 scenario, all members (consumers and prosumers) are organized into a **single Renewable Energy Community (REC)** that spans across both balancing groups. Before supplier billing, the REC performs internal energy settlement to maximize self-consumption and reduce external grid dependency.

**Settlement Order:**
1. Day-Ahead Market → Supplier purchases based on aggregated forecasts
2. Intraday Market → Supplier adjusts position based on updated forecasts  
3. **REC Internal Settlement** → Community shares energy internally (THIS STEP)
4. Balancing Market → Supplier handles remaining imbalances
5. Supplier Billing → Members billed only for residual energy

**REC Structure:**
- **REC_COMBINED**: Single REC containing all 15 members
  - 9 consumers (Supplier A / BG_SUP_A_001)
  - 6 prosumers with PV (Supplier B / BG_SUP_B_001)
  - Members retain their original supplier and balancing group
  - Internal sharing occurs across both balancing groups

### 3.2 REC Settlement Algorithm

For each REC at each timestamp:

1. Calculate total generation: $\text{Gen}_{\text{total}} = \sum_{i \in \text{REC}} \text{RES}_i$

2. Calculate total load: $\text{Load}_{\text{total}} = \sum_{i \in \text{REC}} \text{Load}_i$

3. Apply meter corrections based on comparison:

**Case 1: Perfect Match** ($\text{Gen}_{\text{total}} = \text{Load}_{\text{total}}$)
- All generation consumed internally
- All load satisfied internally
- External grid exchange = 0

$$
\text{Load}_i^{\text{corrected}} = 0 \quad \forall i
$$

$$
\text{RES}_i^{\text{corrected}} = 0 \quad \forall i
$$

**Case 2: Deficit** ($\text{Gen}_{\text{total}} < \text{Load}_{\text{total}}$)
- All generation consumed internally
- Remaining load purchased from grid

$$
\text{RES}_i^{\text{corrected}} = 0 \quad \forall i
$$

$$
\text{Load}_i^{\text{corrected}} = \text{Load}_i - \frac{\text{Load}_i}{\text{Load}_{\text{total}}} \times \text{Gen}_{\text{total}}
$$

**Case 3: Surplus** ($\text{Gen}_{\text{total}} > \text{Load}_{\text{total}}$)
- All load satisfied internally
- Remaining generation sold to grid

$$
\text{Load}_i^{\text{corrected}} = 0 \quad \forall i
$$

$$
\text{RES}_i^{\text{corrected}} = \text{RES}_i - \frac{\text{RES}_i}{\text{Gen}_{\text{total}}} \times \text{Load}_{\text{total}}
$$

**Edge Case Handling:**
If proportional share exceeds individual meter reading (would result in negative), cap at zero and redistribute remainder to other members.

In [ ]:
def calculate_rec_settlement(config, es_timeseries_df, load_actual_df, gen_actual_df):
    """
    Calculate REC internal settlement and add corrected meter readings.
    
    This function:
    1. Identifies which balancing groups belong to RECs
    2. For each REC, collects member load and generation data
    3. Calculates internal energy sharing (min of total generation and total load)
    4. Proportionally reduces each member's meters by their share of shared energy
    5. Aggregates corrected values back to balancing group level
    6. Returns es_timeseries_df with additional columns:
       - net_load: actual_load - actual_gen (before REC settlement)
       - rec_id: REC identifier
       - internal_shared_energy_mwh: amount of energy shared within REC
       - corrected_net_load: net load after internal sharing
    
    Parameters:
    -----------
    config : dict
        Configuration dictionary with prosumers, consumers, and recs
    es_timeseries_df : pd.DataFrame
        Energy system timeseries with supplier and balancing group info
    load_actual_df : pd.DataFrame
        Actual load data by member
    gen_actual_df : pd.DataFrame
        Actual generation data by member
    
    Returns:
    --------
    tuple: (es_timeseries_df, corrected_load_df, corrected_gen_df)
        - es_timeseries_df: Updated with REC settlement columns
        - corrected_load_df: Member-level corrected load after REC sharing
        - corrected_gen_df: Member-level corrected generation after REC sharing
    """
    # Build mapping of balancing group to REC by iterating through members
    bg_to_rec = {}
    
    # Check prosumers for REC assignments
    for prosumer in config['prosumers']:
        bg_id = prosumer['supplier']['balancing_group_id']
        rec_id = prosumer.get('rec', '')
        if rec_id:
            bg_to_rec[bg_id] = rec_id
    
    # Check consumers for REC assignments
    for consumer in config['consumers']:
        bg_id = consumer['supplier']['balancing_group_id']
        rec_id = consumer.get('rec', '')
        if rec_id:
            bg_to_rec[bg_id] = rec_id
    
    # Initialize new columns in desired order
    es_timeseries_df['net_load'] = 0.0
    es_timeseries_df['rec_id'] = ''
    es_timeseries_df['internal_shared_energy_mwh'] = 0.0
    es_timeseries_df['corrected_net_load'] = 0.0
    
    # Initialize corrected dataframes with original values (for non-REC members)
    corrected_load_df = load_actual_df.copy()
    corrected_gen_df = gen_actual_df.copy()
    
    # First, calculate original net_load for each BG (before REC)
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Aggregate actuals for this balancing group
            bg_load_actual = pd.Series(0, index=load_actual_df.index)
            bg_gen_actual = pd.Series(0, index=gen_actual_df.index)
            
            # Aggregate prosumers
            for prosumer in config['prosumers']:
                if prosumer['supplier']['supplier_id'] == supplier_id and \
                   prosumer['supplier']['balancing_group_id'] == bg_id:
                    
                    # Prosumers might have load (not in this config) and generation
                    if 'res' in prosumer and prosumer['res']:
                        gen_id = prosumer['res']['id']
                        if gen_id in gen_actual_df.columns:
                            bg_gen_actual += gen_actual_df[gen_id]
            
            # Aggregate consumers
            for consumer in config['consumers']:
                if consumer['supplier']['supplier_id'] == supplier_id and \
                   consumer['supplier']['balancing_group_id'] == bg_id:
                    
                    load_id = consumer['load']['id']
                    if load_id in load_actual_df.columns:
                        bg_load_actual += load_actual_df[load_id]
            
            # Calculate net load (load - gen, positive = net load)
            bg_net_load = bg_load_actual - bg_gen_actual
            
            # Update es_timeseries_df for this BG
            mask = (es_timeseries_df['supplier_id'] == supplier_id) & \
                   (es_timeseries_df['balancing_group_id'] == bg_id)
            
            for idx in es_timeseries_df[mask].index:
                timestamp = es_timeseries_df.at[idx, 'datetime']
                es_timeseries_df.at[idx, 'net_load'] = bg_net_load.loc[timestamp]
    
    # Process each REC
    for rec in config['recs']:
        rec_id = rec['rec_id']
        
        # Build column names for this REC's members by checking which members belong to this REC
        rec_load_cols = []
        rec_gen_cols = []
        
        # Check prosumers
        for prosumer in config['prosumers']:
            if prosumer.get('rec', '') == rec_id:
                # Prosumers have generation
                if 'res' in prosumer and prosumer['res']:
                    gen_id = prosumer['res']['id']
                    rec_gen_cols.append(gen_id)
        
        # Check consumers
        for consumer in config['consumers']:
            if consumer.get('rec', '') == rec_id:
                # Consumers have load
                load_id = consumer['load']['id']
                rec_load_cols.append(load_id)
        
        # Calculate REC totals (vectorized) - use actual column names from dataframes
        gen_total = gen_actual_df[[col for col in rec_gen_cols if col in gen_actual_df.columns]].sum(axis=1)
        load_total = load_actual_df[[col for col in rec_load_cols if col in load_actual_df.columns]].sum(axis=1)
        
        # Internal shared energy is the minimum of generation and load
        shared_energy = np.minimum(gen_total, load_total)
        
        # Calculate corrected meters for each member (proportional sharing)
        # Update the corrected dataframes in place
        for load_col in [col for col in rec_load_cols if col in load_actual_df.columns]:
            original_load = load_actual_df[load_col]
            # Fraction of total load this member represents
            share_fraction = np.where(load_total > 0, original_load / load_total, 0)
            # Reduce by proportional share of shared energy
            reduction = share_fraction * shared_energy
            corrected_load_df[load_col] = original_load - reduction
        
        for gen_col in [col for col in rec_gen_cols if col in gen_actual_df.columns]:
            original_gen = gen_actual_df[gen_col]
            # Fraction of total generation this member represents
            share_fraction = np.where(gen_total > 0, original_gen / gen_total, 0)
            # Reduce by proportional share of shared energy
            reduction = share_fraction * shared_energy
            corrected_gen_df[gen_col] = original_gen - reduction
        
        # Now aggregate corrected values back to balancing group level
        for supplier in config['suppliers']:
            supplier_id = supplier['supplier_id']
            
            for bg in supplier['balancing_groups']:
                bg_id = bg['balancing_group_id']
                
                # Check if this BG belongs to this REC
                if bg_to_rec.get(bg_id, '') == rec_id:
                    # Find members in this BG that are in this REC
                    bg_member_load_cols = []
                    bg_member_gen_cols = []
                    
                    # Check prosumers
                    for prosumer in config['prosumers']:
                        if prosumer['supplier']['balancing_group_id'] == bg_id and prosumer.get('rec', '') == rec_id:
                            if 'res' in prosumer and prosumer['res']:
                                gen_id = prosumer['res']['id']
                                bg_member_gen_cols.append(gen_id)
                    
                    # Check consumers
                    for consumer in config['consumers']:
                        if consumer['supplier']['balancing_group_id'] == bg_id and consumer.get('rec', '') == rec_id:
                            load_id = consumer['load']['id']
                            bg_member_load_cols.append(load_id)
                    
                    # Filter to only existing columns
                    bg_member_load_cols = [col for col in bg_member_load_cols if col in corrected_load_df.columns]
                    bg_member_gen_cols = [col for col in bg_member_gen_cols if col in corrected_gen_df.columns]
                    
                    # Get original load and gen for this BG
                    if bg_member_load_cols:
                        original_load_bg = load_actual_df[bg_member_load_cols].sum(axis=1)
                        corrected_load_bg = corrected_load_df[bg_member_load_cols].sum(axis=1)
                    else:
                        original_load_bg = pd.Series(0, index=load_actual_df.index)
                        corrected_load_bg = pd.Series(0, index=corrected_load_df.index)
                    
                    if bg_member_gen_cols:
                        original_gen_bg = gen_actual_df[bg_member_gen_cols].sum(axis=1)
                        corrected_gen_bg = corrected_gen_df[bg_member_gen_cols].sum(axis=1)
                    else:
                        original_gen_bg = pd.Series(0, index=gen_actual_df.index)
                        corrected_gen_bg = pd.Series(0, index=corrected_gen_df.index)
                    
                    # Calculate shared energy for this BG (reduction in load or gen)
                    shared_load = original_load_bg - corrected_load_bg
                    shared_gen = original_gen_bg - corrected_gen_bg
                    # Shared energy is whichever is non-zero (load reduction for consumers, gen reduction for prosumers)
                    shared_energy_bg = shared_load + shared_gen
                    
                    # Calculate corrected net load
                    corrected_net_load_series = corrected_load_bg - corrected_gen_bg
                    
                    # Update es_timeseries_df for this BG
                    mask = (es_timeseries_df['supplier_id'] == supplier_id) & \
                           (es_timeseries_df['balancing_group_id'] == bg_id)
                    
                    for idx in es_timeseries_df[mask].index:
                        timestamp = es_timeseries_df.at[idx, 'datetime']
                        es_timeseries_df.at[idx, 'rec_id'] = rec_id
                        
                        # Set internal shared energy first
                        es_timeseries_df.at[idx, 'internal_shared_energy_mwh'] = shared_energy_bg.loc[timestamp]
                        
                        # Then set corrected net load
                        es_timeseries_df.at[idx, 'corrected_net_load'] = corrected_net_load_series.loc[timestamp]
    
    return es_timeseries_df, corrected_load_df, corrected_gen_df

# Apply REC settlement
es_timeseries_df, corrected_load_df, corrected_gen_df = calculate_rec_settlement(
    config, 
    es_timeseries_df,
    es_data['load_actual'],
    es_data['res_actual']
)
es_timeseries_df.head()

### 3.3 REC Settlement Flow - Data Transformation

**Data Flow Summary:**

```
┌─────────────────────────────────────────────────────────────────────────┐
│ STEP 1: ORIGINAL METER READINGS (from SimBench data)                   │
├─────────────────────────────────────────────────────────────────────────┤
│ es_data['load_actual']  →  Original load for all members               │
│ es_data['res_actual']   →  Original generation for all members         │
└─────────────────────────────────────────────────────────────────────────┘
                                    ↓
┌─────────────────────────────────────────────────────────────────────────┐
│ STEP 2: REC INTERNAL SETTLEMENT                                         │
├─────────────────────────────────────────────────────────────────────────┤
│ Function: calculate_rec_settlement()                                    │
│                                                                         │
│ For each REC at each timestamp:                                        │
│   1. Total generation: Σ(all prosumer PV)                              │
│   2. Total load: Σ(all member consumption)                             │
│   3. Internal sharing: min(total_gen, total_load)                      │
│   4. Reduce meters proportionally by share_fraction                    │
│                                                                         │
│ Output:                                                                 │
│   • corrected_load_df  = load - (load_fraction × shared_energy)       │
│   • corrected_gen_df   = gen - (gen_fraction × shared_energy)         │
│   • es_timeseries_df['internal_shared_energy_mwh']                    │
└─────────────────────────────────────────────────────────────────────────┘
                                    ↓
┌─────────────────────────────────────────────────────────────────────────┐
│ STEP 3: DOWNSTREAM CALCULATIONS (use corrected values)                 │
├─────────────────────────────────────────────────────────────────────────┤
│ ✅ Balancing Market:                                                    │
│    calculate_balancing_market_positions(corrected_load_df,             │
│                                        corrected_gen_df, ...)          │
│    → Creates: actual_load_mwh, actual_gen_mwh, imbalances             │
│                                                                         │
│ ✅ Customer Billing:                                                    │
│    calculate_customer_bills(corrected_load_df, corrected_gen_df, ...) │
│    → Creates: sales_revenue_eur, purchase_costs_eur                   │
│                                                                         │
│ ✅ Financial Analysis:                                                  │
│    All revenue/cost calculations based on corrected grid exchange      │
└─────────────────────────────────────────────────────────────────────────┘
```

**Key Insight:** 
The terms "actual_load_mwh" and "actual_gen_mwh" in results refer to **actual grid exchange** (corrected values), NOT original meter readings. This reflects what the supplier actually sees and bills for.

## 4. Balancing Market Operations

### 4.1 Calculate Actual Imbalances

Compare actual consumption/generation against intra-day forecasts to determine balancing requirements.

**IMPORTANT - REC Settlement Impact:**

After REC internal settlement, all downstream calculations use **CORRECTED meter readings** instead of original values:

✅ **Corrected Load (`corrected_load_df`)**: Original load reduced by member's share of internal sharing
✅ **Corrected Generation (`corrected_gen_df`)**: Original generation reduced by member's share of internal sharing

**What This Means:**
- `actual_load_mwh` in results = **corrected load** (after REC sharing)
- `actual_gen_mwh` in results = **corrected generation** (after REC sharing)
- **Balancing imbalances** = calculated from corrected values
- **Customer billing** = based on corrected grid import/export
- **Supplier wholesale positions** = reflect reduced grid dependency

**Example:**
```
Original: Prosumer generates 10 MWh, consumes 3 MWh → exports 7 MWh to grid
REC shares: 5 MWh consumed internally by other members
Corrected: Prosumer generates 5 MWh, consumes 0 MWh → exports 5 MWh to grid
Supplier only sees: 5 MWh generation (not 10 MWh)
```

This ensures the financial calculations reflect the **actual grid exchange** after community self-consumption.

In [ ]:
def calculate_balancing_market_positions(es_timeseries_df, load_actual_df, gen_actual_df, prices_df, config):
    """
    Calculate balancing market positions, imbalances, and settlements using single imbalance price
    
    IMPORTANT: For REC scenarios, this function should receive CORRECTED meter readings
    (corrected_load_df, corrected_gen_df) that reflect the state AFTER REC internal settlement.
    
    Process:
    1. Aggregate actual load and generation by supplier and balancing group
    2. Calculate BRP imbalances (Actual - ID Forecast)
    3. Apply single imbalance price with sign-based penalty/reward logic
    4. Add balancing columns to es_timeseries_df
    
    Balancing Group Perspective:
    - actual_load_mwh: Actual consumption (always ≥ 0)
    - actual_gen_mwh: Actual generation (always ≥ 0)
    - balancing_group_actual_mwh: Net position (load - gen), positive = deficit
    - balancing_group_forecast_mwh: Forecasted net position from ID market
    - imbalance_mwh: Deviation from forecast (actual - forecast)
    
    Imbalance Sign Convention:
    - POSITIVE imbalance: Actual deficit > forecast (need to buy more from balancing market)
    - NEGATIVE imbalance: Actual deficit < forecast (have surplus to sell to balancing market)
    
    Imbalance Price Sign Convention:
    - NEGATIVE imbalance price: System has EXCESS generation (supply > demand)
      * Generators are PENALIZED (receive negative price)
      * Consumers are REWARDED (pay negative price = receive money)
    
    - POSITIVE imbalance price: System has DEFICIT (supply < demand)
      * Generators are REWARDED (receive positive price)
      * Consumers are PENALIZED (pay positive price)
    
    Settlement Logic:
    Settlement = Imbalance × Imbalance_Price
    
    Returns updated es_timeseries_df with balancing columns added
    """
    # Step 1: Aggregate actual positions by supplier and balancing group
    balancing_data_list = []
    
    for supplier in config['suppliers']:
        supplier_id = supplier['supplier_id']
        
        for bg in supplier['balancing_groups']:
            bg_id = bg['balancing_group_id']
            
            # Initialize aggregated actuals
            bg_load_actual = pd.Series(0, index=load_actual_df.index)
            bg_gen_actual = pd.Series(0, index=gen_actual_df.index)
            
            # Aggregate prosumers in this balancing group
            for prosumer in config['prosumers']:
                if prosumer['supplier']['supplier_id'] == supplier_id and \
                   prosumer['supplier']['balancing_group_id'] == bg_id:
                    
                    # Add load actual if exists
                    if 'load' in prosumer and prosumer['load']:
                        load_id = prosumer['load']['id']
                        if load_id in load_actual_df.columns:
                            bg_load_actual += load_actual_df[load_id]
                    
                    # Add generation actual if exists
                    if 'res' in prosumer and prosumer['res']:
                        gen_id = prosumer['res']['id']
                        if gen_id in gen_actual_df.columns:
                            bg_gen_actual += gen_actual_df[gen_id]
            
            # Aggregate consumers in this balancing group
            for consumer in config['consumers']:
                if consumer['supplier']['supplier_id'] == supplier_id and \
                   consumer['supplier']['balancing_group_id'] == bg_id:
                    
                    load_id = consumer['load']['id']
                    if load_id in load_actual_df.columns:
                        bg_load_actual += load_actual_df[load_id]
            
            # Get ID net positions (closing commitments) from es_timeseries_df
            id_data = es_timeseries_df[
                (es_timeseries_df['supplier_id'] == supplier_id) & 
                (es_timeseries_df['balancing_group_id'] == bg_id)
            ].set_index('datetime')
            
            id_net_load_forecast = id_data['id_net_load_forecast_mwh']
            id_net_gen_forecast = id_data['id_net_gen_forecast_mwh']
            # Balancing Group Perspective: Supplier buys from market to serve customers
            # Actual position: positive = deficit (need to buy), negative = surplus (can sell)
            balancing_group_actual_mwh = bg_load_actual - bg_gen_actual
            
            # Forecast position from ID market closing
            balancing_group_forecast_mwh = id_net_load_forecast - id_net_gen_forecast
            
            # Imbalance: Actual - Forecast
            # Positive = actual deficit > forecasted deficit (need to buy more)
            # Negative = actual deficit < forecasted deficit (have surplus to sell)
            imbalance_mwh = balancing_group_actual_mwh - balancing_group_forecast_mwh
            
            # Get imbalance price (sign indicates system position)
            imbalance_price = prices_df['imbalance_price']
            
            # Calculate settlement
            balancing_settlement = imbalance_mwh * imbalance_price
            
            # Split settlement into penalty and reward (both as positive amounts)
            # Penalty: when settlement is negative (BRP must pay)
            # Reward: when settlement is positive (BRP receives payment)
            imbalance_penalty = balancing_settlement.apply(lambda x: abs(x) if x < 0 else 0)
            imbalance_reward = balancing_settlement.apply(lambda x: x if x > 0 else 0)
            
            # Store time series data for merging
            for timestamp in load_actual_df.index:
                balancing_data_list.append({
                    'datetime': timestamp,
                    'supplier_id': supplier_id,
                    'balancing_group_id': bg_id,
                    'balancing_group_actual_mwh': balancing_group_actual_mwh.loc[timestamp],
                    'balancing_group_forecast_mwh': balancing_group_forecast_mwh.loc[timestamp],
                    'imbalance_mwh': imbalance_mwh.loc[timestamp],
                    'imbalance_price_eur_per_mwh': imbalance_price.loc[timestamp],
                    'imbalance_penalty': imbalance_penalty.loc[timestamp],
                    'imbalance_reward': imbalance_reward.loc[timestamp]
                })
    
    # Create temporary DataFrame with balancing data
    balancing_data_df = pd.DataFrame(balancing_data_list)
    
    # Drop any columns from balancing_data_df that already exist in es_timeseries_df
    # Keep only merge keys and new columns
    merge_keys = ['datetime', 'supplier_id', 'balancing_group_id']
    shared_columns = [col for col in balancing_data_df.columns 
                     if col in es_timeseries_df.columns and col not in merge_keys]
    
    if shared_columns:
        balancing_data_df = balancing_data_df.drop(columns=shared_columns)
    
    # Merge balancing columns into es_timeseries_df
    es_timeseries_df = es_timeseries_df.merge(
        balancing_data_df,
        on=merge_keys,
        how='left'
    )
    
    return es_timeseries_df

# Calculate balancing market positions using CORRECTED meter readings after REC settlement
es_timeseries_df = calculate_balancing_market_positions(
    es_timeseries_df,
    corrected_load_df,  # Use corrected load after REC sharing
    corrected_gen_df,   # Use corrected generation after REC sharing
    es_data['prices'],
    config
)

# Display sample of time series data
display(es_timeseries_df.head())

In [ ]:
# VERIFICATION: Confirm that corrected values are used in balancing market
print("="*80)
print("VERIFICATION: Corrected vs Original Values After REC Settlement")
print("="*80)

# Compare original vs corrected for a sample prosumer
sample_prosumer = config['prosumers'][0]
sample_gen_id = sample_prosumer['res']['id']

if sample_gen_id in es_data['res_actual'].columns:
    original_gen = es_data['res_actual'][sample_gen_id]
    corrected_gen = corrected_gen_df[sample_gen_id]
    
    print(f"\nSample Prosumer: {sample_prosumer['meter_id']}")
    print(f"Generation ID: {sample_gen_id}")
    print(f"\nFirst 5 timestamps:")
    print(f"  Original Generation (MWh):  {original_gen.head().values}")
    print(f"  Corrected Generation (MWh): {corrected_gen.head().values}")
    print(f"  Reduction (MWh):            {(original_gen - corrected_gen).head().values}")
    
    total_original = original_gen.sum()
    total_corrected = corrected_gen.sum()
    reduction_pct = ((total_original - total_corrected) / total_original * 100) if total_original > 0 else 0
    
    print(f"\nAnnual Totals:")
    print(f"  Original:  {total_original:,.2f} MWh")
    print(f"  Corrected: {total_corrected:,.2f} MWh")
    print(f"  Shared internally: {total_original - total_corrected:,.2f} MWh ({reduction_pct:.1f}%)")

# Verify that es_timeseries_df contains corrected values
print(f"\n" + "-"*80)
print("Verification: es_timeseries_df contains CORRECTED values")
print("-"*80)

# Get balancing group data for the sample prosumer's supplier
sample_supplier_id = sample_prosumer['supplier']['supplier_id']
sample_bg_id = sample_prosumer['supplier']['balancing_group_id']

bg_data = es_timeseries_df[
    (es_timeseries_df['supplier_id'] == sample_supplier_id) & 
    (es_timeseries_df['balancing_group_id'] == sample_bg_id)
].head()

print(f"\nBalancing Group: {sample_supplier_id} / {sample_bg_id}")
print(f"Columns in es_timeseries_df with balancing calculations:")
print(f"  - balancing_group_actual_mwh: {bg_data['balancing_group_actual_mwh'].values[:3]}")
print(f"  - balancing_group_forecast_mwh: {bg_data['balancing_group_forecast_mwh'].values[:3]}")
print(f"  - imbalance_mwh: {bg_data['imbalance_mwh'].values[:3]}")
print(f"  - imbalance_price_eur_per_mwh: {bg_data['imbalance_price_eur_per_mwh'].values[:3]}")
print(f"  - imbalance_penalty: {bg_data['imbalance_penalty'].values[:3]}")
print(f"  - imbalance_reward: {bg_data['imbalance_reward'].values[:3]}")

print(f"\n✅ Confirmation: Balancing calculations are based on CORRECTED meter readings")
print(f"✅ REC internal sharing has been applied BEFORE balancing market calculation")
print("="*80)

### 4.2 Balancing Market - Mathematical Formulation

This section documents the mathematical formulation of the **dual-pricing balancing mechanism** implemented in the `calculate_balancing_market_positions()` function.

---

#### **Column Definitions and Mathematical Formulas**

The balancing market calculation creates the following columns in `es_timeseries_df`:

##### **1. Actual Net Position** (`actual_net_mwh`)

$$
Q_{\text{actual,net}}^{t} = Q_{\text{RES,actual}}^{t} - Q_{\text{load,actual}}^{t}
$$

Where:
- $Q_{\text{actual,net}}^{t}$ = Net position at timestamp $t$ (MWh)
- $Q_{\text{RES,actual}}^{t}$ = Actual renewable generation at timestamp $t$ (MWh)
- $Q_{\text{load,actual}}^{t}$ = Actual load consumption at timestamp $t$ (MWh)

**Sign Convention:**
- **Positive**: Surplus (generation > consumption)
- **Negative**: Deficit (generation < consumption)

---

##### **2. BRP Imbalance** (`imbalance_mwh`)

$$
\Delta Q_{\text{BRP}}^{t} = Q_{\text{actual,net}}^{t} - Q_{\text{ID,forecast}}^{t}
$$

Where:
- $\Delta Q_{\text{BRP}}^{t}$ = BRP imbalance at timestamp $t$ (MWh)
- $Q_{\text{ID,forecast}}^{t}$ = Intra-day net forecast at timestamp $t$ (MWh)

**Sign Convention:**
- **Positive imbalance** ($\Delta Q_{\text{BRP}}^{t} > 0$): BRP is **LONG** (surplus position)
  * More energy available than committed
  
- **Negative imbalance** ($\Delta Q_{\text{BRP}}^{t} < 0$): BRP is **SHORT** (deficit position)
  * Less energy available than committed

---

##### **3. System Imbalance Indicator** (`system_imbalance_indicator`)

$$
I_{\text{system}}^{t} = 
\begin{cases}
-1 & \text{if system is SHORT (needs power)} \\
0 & \text{if system is BALANCED} \\
+1 & \text{if system has EXCESS (surplus power)}
\end{cases}
$$

This indicator represents the **TSO-level system imbalance** direction:
- $I_{\text{system}}^{t} = -1$: System shortage → TSO needs more power
- $I_{\text{system}}^{t} = 0$: System balanced
- $I_{\text{system}}^{t} = +1$: System excess → TSO has surplus power

---

##### **4. Helps System Flag** (`helps_system`)

$$
\text{helps\_system}^{t} = 
\begin{cases}
\text{True} & \text{if } \Delta Q_{\text{BRP}}^{t} \times I_{\text{system}}^{t} < 0 \\
\text{False} & \text{otherwise}
\end{cases}
$$

This boolean flag indicates whether the BRP's imbalance **helps** or **worsens** the system imbalance:

| BRP Imbalance | System Status | Product Sign | BRP Action | Result |
|---------------|---------------|--------------|------------|---------|
| $+$ (surplus) | $-1$ (short) | Negative | **Helps** | Reward |
| $-$ (deficit) | $-1$ (short) | Positive | Worsens | Penalty |
| $+$ (surplus) | $+1$ (excess) | Positive | Worsens | Penalty |
| $-$ (deficit) | $+1$ (excess) | Negative | **Helps** | Reward |

---

##### **5. Penalty Price** (`penalty_price_eur_per_mwh`)

$$
p_{\text{penalty}}^{t} = \text{penalty\_price}^{t} \quad \text{(EUR/MWh)}
$$

Price applied when BRP **worsens** system imbalance (same sign).

---

##### **6. Reward Price** (`reward_price_eur_per_mwh`)

$$
p_{\text{reward}}^{t} = \text{reward\_price}^{t} \quad \text{(EUR/MWh)}
$$

Price applied when BRP **helps** system imbalance (opposite sign). Typically: $p_{\text{reward}}^{t} < p_{\text{penalty}}^{t}$

---

##### **7. Net Reward/Penalty Settlement** (`net_reward_penalty_eur`)

This is the **primary settlement column** combining all balancing costs/revenues:

$$
S_{\text{balancing}}^{t} = 
\begin{cases}
0 & \text{if } \Delta Q_{\text{BRP}}^{t} = 0 \\
+|\Delta Q_{\text{BRP}}^{t}| \times p_{\text{reward}}^{t} & \text{if } \Delta Q_{\text{BRP}}^{t} \times I_{\text{system}}^{t} < 0 \quad \text{(BRP helps)} \\
-|\Delta Q_{\text{BRP}}^{t}| \times p_{\text{penalty}}^{t} & \text{if } \Delta Q_{\text{BRP}}^{t} \times I_{\text{system}}^{t} > 0 \quad \text{(BRP worsens)}
\end{cases}
$$

**Economic Sign Convention:**
- **Positive** ($S_{\text{balancing}}^{t} > 0$): **REWARD** → BRP receives payment (revenue/credit)
- **Negative** ($S_{\text{balancing}}^{t} < 0$): **PENALTY** → BRP pays cost (cost/debit)

---

#### **Complete Settlement Formula**

##### **8. Final Settlement** (`final_settlement_eur`)

$$
S_{\text{final}}^{t} = S_{\text{DA}}^{t} + S_{\text{ID}}^{t} + S_{\text{balancing}}^{t}
$$

Where:
- $S_{\text{DA}}^{t}$ = Day-ahead net commitment (EUR)
- $S_{\text{ID}}^{t}$ = Intra-day net adjustment (EUR)
- $S_{\text{balancing}}^{t}$ = Balancing reward/penalty (EUR)

**Total Supplier Wholesale Cost/Revenue:**

$$
C_{\text{total}} = \sum_{t} S_{\text{final}}^{t}
$$

- **Negative total**: Net cost to supplier (money out)
- **Positive total**: Net revenue to supplier (money in)

---

#### **Dual-Pricing Mechanism Logic**

The dual-pricing mechanism incentivizes BRPs to help balance the system:

| Scenario | BRP Imbalance | System Status | Price Applied | Settlement |
|----------|---------------|---------------|---------------|------------|
| **1** | BRP surplus (long)<br>$\Delta Q_{\text{BRP}} > 0$ | System short<br>$I = -1$ | $p_{\text{reward}}$ | $+\|\Delta Q\| \times p_{\text{reward}}$ |
| **2** | BRP deficit (short)<br>$\Delta Q_{\text{BRP}} < 0$ | System short<br>$I = -1$ | $p_{\text{penalty}}$ | $-\|\Delta Q\| \times p_{\text{penalty}}$ |
| **3** | BRP deficit (short)<br>$\Delta Q_{\text{BRP}} < 0$ | System excess<br>$I = +1$ | $p_{\text{reward}}$ | $+\|\Delta Q\| \times p_{\text{reward}}$ |
| **4** | BRP surplus (long)<br>$\Delta Q_{\text{BRP}} > 0$ | System excess<br>$I = +1$ | $p_{\text{penalty}}$ | $-\|\Delta Q\| \times p_{\text{penalty}}$ |

**Key Principle:** 
- When BRP and system have **opposite signs** → BRP **helps** → **Reward** (positive EUR)
- When BRP and system have **same sign** → BRP **worsens** → **Penalty** (negative EUR)

---

#### **Implementation Notes**

1. **Aggregation Level**: Calculations performed at **balancing group** level for each supplier
2. **Time Resolution**: 15-minute intervals (35,136 timestamps per year)
3. **Price Sources**: 
   - Penalty/reward prices from `prices.csv`
   - System imbalance indicator from `system_imbalance_balance_indicator.csv`
4. **Data Flow**:
   ```
   Actual Data → Aggregate by BG → Calculate Imbalance → 
   Check System Status → Apply Dual Pricing → Generate Settlement
   ```

---


## 5. Supplier Billing to Customers

### 5.1 Calculate Customer Bills

Calculate retail billing for each customer based on retail prices and their actual consumption.

**CRITICAL: Uses CORRECTED Meter Readings**

Customer billing is calculated using **corrected meter readings** after REC internal settlement:

- **Prosumers:** Billed only for **residual grid import** (after internal sharing reduces their load)
- **Prosumers:** Paid only for **residual grid export** (after internal sharing reduces their generation)
- **Consumers:** Billed only for **residual grid import** (after internal sharing reduces their load)

**Financial Impact:**
```
Without REC: Prosumer exports 10 MWh → Supplier pays 10 × €82.40 = €824
With REC:    Prosumer exports 5 MWh → Supplier pays 5 × €82.40 = €412 (50% saving)
```

The REC reduces supplier costs by minimizing grid transactions.

In [ ]:
def calculate_customer_bills(config, load_actual_df, gen_actual_df, prices_df):
    """
    Calculate customer-level billing with retail prices.
    NOTE: This function now receives CORRECTED meter readings after REC internal settlement.
    
    Args:
        config: Configuration dictionary with prosumers and consumers
        load_actual_df: Actual load data (corrected after REC settlement)
        gen_actual_df: Actual generation data (corrected after REC settlement)
        prices_df: Prices dataframe with retail_price and feedin_price columns
    
    Returns:
        pd.DataFrame: Customer-level billing dataframe with columns:
            - datetime, supplier_id, balancing_group_id, customer_id, customer_type
            - actual_load_mwh, actual_gen_mwh, net_load_mwh
            - retail_price_eur_per_mwh, feedin_price_eur_per_mwh
            - sales_revenue_eur, purchase_costs_eur
    """
    retail_price = prices_df['retail_price']
    feedin_price = prices_df['feedin_price']
    
    all_records = []
    
    # Process prosumers (load + generation)
    for prosumer in config['prosumers']:
        customer_id = prosumer['meter_id']
        supplier_id = prosumer['supplier']['supplier_id']
        bg_id = prosumer['supplier']['balancing_group_id']
        
        # Get actual generation
        gen_id = prosumer['res']['id']
        if gen_id in gen_actual_df.columns:
            actual_gen = gen_actual_df[gen_id]
        else:
            # Fallback: use first available generation column
            gen_cols = [col for col in gen_actual_df.columns if col != 'datetime']
            if gen_cols:
                actual_gen = gen_actual_df[gen_cols[0]]
                print(f"Warning: Generation ID '{gen_id}' not found for {customer_id}, using '{gen_cols[0]}'")
            else:
                actual_gen = pd.Series(0.0, index=gen_actual_df.index)
        
        # Get actual load (if prosumer has load)
        actual_load = None
        if 'load' in prosumer:
            load_id = prosumer['load']['id']
            if load_id in load_actual_df.columns:
                actual_load = load_actual_df[load_id]
            else:
                # Fallback: use first available load column
                load_cols = [col for col in load_actual_df.columns if col != 'datetime']
                if load_cols:
                    actual_load = load_actual_df[load_cols[0]]
                    print(f"Warning: Load ID '{load_id}' not found for {customer_id}, using '{load_cols[0]}'")
        
        if actual_load is None:
            actual_load = pd.Series(0.0, index=gen_actual_df.index)
        
        # Net load (positive = import, negative = export)
        net_load = actual_load - actual_gen
        grid_import = net_load.clip(lower=0)
        grid_export = (-net_load).clip(lower=0)
        
        # Calculate billing
        sales_revenue = grid_import * retail_price  # Supplier sells to customer
        purchase_costs = grid_export * feedin_price  # Supplier buys from customer
        
        # Ensure no negative zeros
        sales_revenue = sales_revenue.abs()
        purchase_costs = purchase_costs.abs()
        
        # Create dataframe for this customer
        customer_df = pd.DataFrame({
            'datetime': prices_df.index,
            'supplier_id': supplier_id,
            'balancing_group_id': bg_id,
            'customer_id': customer_id,
            'customer_type': 'prosumer',
            'actual_load_mwh': actual_load.values,
            'actual_gen_mwh': actual_gen.values,
            'net_load_mwh': net_load.values,
            'retail_price_eur_per_mwh': retail_price.values,
            'feedin_price_eur_per_mwh': feedin_price.values,
            'sales_revenue_eur': sales_revenue.values,
            'purchase_costs_eur': purchase_costs.values
        })
        
        all_records.append(customer_df)
    
    # Process consumers (only load, no RES)
    for consumer in config['consumers']:
        customer_id = consumer['meter_id']
        supplier_id = consumer['supplier']['supplier_id']
        bg_id = consumer['supplier']['balancing_group_id']
        
        # Get actual load
        load_id = consumer['load']['id']
        if load_id in load_actual_df.columns:
            actual_load = load_actual_df[load_id]
        else:
            # Fallback: use first available load column
            load_cols = [col for col in load_actual_df.columns if col != 'datetime']
            if load_cols:
                actual_load = load_actual_df[load_cols[0]]
                print(f"Warning: Load ID '{load_id}' not found for {customer_id}, using '{load_cols[0]}'")
            else:
                actual_load = pd.Series(0.0, index=load_actual_df.index)
        
        # No generation for consumers
        actual_gen = pd.Series(0.0, index=actual_load.index)
        net_load = actual_load  # All load is imported
        
        # Calculate billing (only sales, no purchases from consumer)
        sales_revenue = actual_load * retail_price
        purchase_costs = pd.Series(0.0, index=actual_load.index)  # Use Series with 0.0 instead of multiplication
        
        # Create dataframe for this customer
        customer_df = pd.DataFrame({
            'datetime': prices_df.index,
            'supplier_id': supplier_id,
            'balancing_group_id': bg_id,
            'customer_id': customer_id,
            'customer_type': 'consumer',
            'actual_load_mwh': actual_load.values,
            'actual_gen_mwh': actual_gen.values,
            'net_load_mwh': net_load.values,
            'retail_price_eur_per_mwh': retail_price.values,
            'feedin_price_eur_per_mwh': feedin_price.values,
            'sales_revenue_eur': sales_revenue.values,
            'purchase_costs_eur': purchase_costs.values
        })
        
        all_records.append(customer_df)
    
    # Concatenate all customer records
    customer_billing_df = pd.concat(all_records, ignore_index=True)
    
    return customer_billing_df

# Calculate customer bills using CORRECTED meter readings
# (after REC internal settlement)
customer_billing_df = calculate_customer_bills(
    config,
    corrected_load_df,  # Use corrected load after REC sharing
    corrected_gen_df,   # Use corrected generation after REC sharing
    es_data['prices']
)


# Display sample of time series data
display(customer_billing_df.head(10))

### 5.2 Aggregate Customer Bills to Balancing Group Level

Aggregate customer billing to balancing group level and merge with wholesale market data.

In [ ]:
# First, aggregate customer billing to balancing group level (timeseries)
retail_bg_df = customer_billing_df.groupby(['datetime', 'supplier_id', 'balancing_group_id']).agg({
    'retail_price_eur_per_mwh': 'mean',
    'feedin_price_eur_per_mwh': 'mean',
    'sales_revenue_eur': 'sum',
    'purchase_costs_eur': 'sum'
}).reset_index()

# Remove any existing retail columns from es_timeseries_df
retail_cols_to_drop = [col for col in es_timeseries_df.columns 
                       if col in ['retail_price_eur_per_mwh', 'feedin_price_eur_per_mwh', 
                                  'sales_revenue_eur', 'purchase_costs_eur'] 
                       or '_x' in col or '_y' in col]
if retail_cols_to_drop:
    es_timeseries_df = es_timeseries_df.drop(columns=retail_cols_to_drop)

# Merge retail billing into es_timeseries_df
es_timeseries_df = es_timeseries_df.merge(
    retail_bg_df,
    on=['datetime', 'supplier_id', 'balancing_group_id'],
    how='left'
)

print(f"\nRetail billing merged into es_timeseries_df")
print(f"   Shape: {es_timeseries_df.shape}")

# Now aggregate combined data to monthly level by balancing group
es_timeseries_copy = es_timeseries_df.copy()
es_timeseries_copy['month_year'] = pd.to_datetime(es_timeseries_copy['datetime']).dt.strftime('%m-%Y')

# Define aggregation logic
numeric_columns = es_timeseries_copy.select_dtypes(include=[np.number]).columns.tolist()
agg_dict = {}
for col in numeric_columns:
    if 'price' in col.lower():
        agg_dict[col] = 'mean'  # Average prices
    else:
        agg_dict[col] = 'sum'   # Sum quantities and currencies

# Aggregate to monthly by balancing group
monthly_bg_df = es_timeseries_copy.groupby(
    ['month_year', 'supplier_id', 'balancing_group_id']
).agg(agg_dict).reset_index()

# Rename month_year to datetime
monthly_bg_df = monthly_bg_df.rename(columns={'month_year': 'datetime'})

print(f"\nMonthly aggregation by balancing group created:")
print(f"   Shape: {monthly_bg_df.shape}")
print(f"   Months: {monthly_bg_df['datetime'].nunique()}")
print(f"   Supplier-BG combinations: {len(monthly_bg_df[['supplier_id', 'balancing_group_id']].drop_duplicates())}")
print(f"\nSample columns: {monthly_bg_df.columns.tolist()[:10]}...")

display(monthly_bg_df.head())

### 5.3 Monthly Aggregation

Aggregate the complete energy system dataframe (with retail billing) to monthly level.

In [ ]:
# Create monthly aggregation for complete energy system dataframe
es_timeseries_df_copy = es_timeseries_df.copy()

# Convert datetime to "MM-YYYY" format (e.g., "01-2016", "02-2016")
es_timeseries_df_copy['month_year'] = pd.to_datetime(es_timeseries_df_copy['datetime']).dt.strftime('%m-%Y')

# Define aggregation logic: mean for prices, sum for quantities and currencies
numeric_columns = es_timeseries_df_copy.select_dtypes(include=[np.number]).columns.tolist()

agg_dict = {}
for col in numeric_columns:
    if 'price' in col.lower():
        agg_dict[col] = 'mean'  # Average prices
    else:
        agg_dict[col] = 'sum'   # Sum quantities and currencies

# Group by month, supplier_id, and balancing_group_id
es_monthly_df = es_timeseries_df_copy.groupby(['month_year', 'supplier_id', 'balancing_group_id']).agg(agg_dict).reset_index()

# Rename month_year to datetime for consistency
es_monthly_df = es_monthly_df.rename(columns={'month_year': 'datetime'})

# Select columns to keep in final output
columns_to_keep = [
    'datetime', 'supplier_id', 'balancing_group_id',
    # Day-ahead market
    'da_load_forecast_mwh', 'da_gen_forecast_mwh',
    'da_purchase_cost_eur', 'da_sale_revenue_eur',
    # Intraday market
    'id_load_adjustment_mwh', 'id_gen_adjustment_mwh', 'id_price_eur_per_mwh',
    'id_purchase_adjustment_eur', 'id_sale_adjustment_eur',
    'closing_purchase_commitment_eur', 'closing_sale_commitment_eur',
    # Balancing market
    'actual_load_mwh', 'actual_gen_mwh',
    'balancing_group_actual_mwh', 'balancing_group_forecast_mwh', 'imbalance_mwh',
    'imbalance_price_eur_per_mwh', 'imbalance_penalty', 'imbalance_reward',
    # REC settlement
    'internal_shared_energy_mwh',
    # Retail billing
    'retail_price_eur_per_mwh', 'feedin_price_eur_per_mwh',
    'sales_revenue_eur', 'purchase_costs_eur'
]

# Keep only the columns that exist in the dataframe
columns_to_keep = [col for col in columns_to_keep if col in es_monthly_df.columns]
es_monthly_df = es_monthly_df[columns_to_keep]

print(f"✅ Monthly aggregation created: {es_monthly_df.shape}")
print(f"   Months: {es_monthly_df['datetime'].nunique()}")
print(f"   Supplier-BG combinations: {es_monthly_df.groupby(['supplier_id', 'balancing_group_id']).ngroups}")
print(f"\n   Columns included: {es_monthly_df.columns.tolist()}")

# Verify retail billing columns
retail_cols = ['sales_revenue_eur', 'purchase_costs_eur']
missing_cols = [col for col in retail_cols if col not in es_monthly_df.columns]
if missing_cols:
    print(f"\n   Missing columns: {missing_cols}")
else:
    print(f"   All retail billing columns present")

display(es_monthly_df)

### 5.4 Calculate Net Wholesale Cost and Retail Profit

Calculate supplier's net wholesale cost and retail profit/loss.

In [ ]:
# Calculate supplier profit/loss
es_monthly_df_analysis = es_monthly_df.copy()

# REVENUES (money coming in)
# 1. Energy market sales
es_monthly_df_analysis['revenue_energy_market_sales_eur'] = es_monthly_df_analysis['closing_sale_commitment_eur']

# 2. Balancing rewards 
es_monthly_df_analysis['revenue_balancing_rewards_eur'] = es_monthly_df_analysis['imbalance_reward']

# 3. Retail sales to customers
es_monthly_df_analysis['revenue_retail_sales_eur'] = es_monthly_df_analysis['sales_revenue_eur']

# Total revenue
es_monthly_df_analysis['total_revenue_eur'] = (
    es_monthly_df_analysis['revenue_energy_market_sales_eur'] +
    es_monthly_df_analysis['revenue_balancing_rewards_eur'] +
    es_monthly_df_analysis['revenue_retail_sales_eur']
)

# COSTS (money going out)
# 1. Energy market purchases
es_monthly_df_analysis['cost_energy_market_purchases_eur'] = es_monthly_df_analysis['closing_purchase_commitment_eur']

# 2. Balancing penalties
es_monthly_df_analysis['cost_balancing_penalties_eur'] = es_monthly_df_analysis['imbalance_penalty']

# 3. Retail purchases from customers
es_monthly_df_analysis['cost_retail_purchases_eur'] = es_monthly_df_analysis['purchase_costs_eur']

# Total costs
es_monthly_df_analysis['total_costs_eur'] = (
    es_monthly_df_analysis['cost_energy_market_purchases_eur'] +
    es_monthly_df_analysis['cost_balancing_penalties_eur'] +
    es_monthly_df_analysis['cost_retail_purchases_eur']
)

# PROFIT/LOSS
es_monthly_df_analysis['profit_loss_eur'] = (
    es_monthly_df_analysis['total_revenue_eur'] - 
    es_monthly_df_analysis['total_costs_eur']
)

print(f"\nFinancial analysis dataframe created: {es_monthly_df_analysis.shape}")
print(f"\n   Revenue Components:")
print(f"   - revenue_energy_market_sales_eur: Energy market sales (DA+ID)")
print(f"   - revenue_balancing_rewards_eur: Balancing market rewards")
print(f"   - revenue_retail_sales_eur: Retail sales to customers")
print(f"\n   Cost Components:")
print(f"   - cost_energy_market_purchases_eur: Energy market purchases (DA+ID)")
print(f"   - cost_balancing_penalties_eur: Balancing market penalties")
print(f"   - cost_retail_purchases_eur: Retail purchases from customers")
print(f"\n   Summary:")
print(f"   - total_revenue_eur: Sum of all revenue components")
print(f"   - total_costs_eur: Sum of all cost components")
print(f"   - profit_loss_eur: Net profit/loss (revenue - costs)")

display(es_monthly_df_analysis[[
    'datetime', 'supplier_id', 'balancing_group_id',
    'revenue_energy_market_sales_eur', 'revenue_balancing_rewards_eur', 'revenue_retail_sales_eur',
    'cost_energy_market_purchases_eur', 'cost_balancing_penalties_eur', 'cost_retail_purchases_eur',
    'total_revenue_eur', 'total_costs_eur', 'profit_loss_eur'
]])

In [ ]:
# Plot monthly financial analysis for both suppliers with SAME SCALES
import matplotlib.pyplot as plt
import numpy as np

# Net imbalance already calculated as imbalance_mwh
# For backward compatibility, create alias
es_monthly_df_analysis['net_imbalance_mwh'] = es_monthly_df_analysis['imbalance_mwh']

# Create supplier name mapping from config
supplier_names = {s['supplier_id']: s['supplier_name'] for s in config['suppliers']}

# Calculate global min/max for consistent scales
all_data = es_monthly_df_analysis[es_monthly_df_analysis['supplier_id'].isin(['SUP_A', 'SUP_B'])]

# Financial overview limits
max_revenue = all_data['total_revenue_eur'].max()
max_costs = all_data['total_costs_eur'].max()
min_profit = all_data['profit_loss_eur'].min()
max_profit = all_data['profit_loss_eur'].max()
financial_ylim = [min(min_profit * 1.1, 0), max(max_revenue, max_costs, max_profit) * 1.1]

# Imbalance limits (combined for both suppliers)
all_imbalances = np.concatenate([
    all_data['balancing_group_actual_mwh'].values,
    all_data['balancing_group_forecast_mwh'].values,
    all_data['imbalance_mwh'].values
])
imb_min = all_imbalances.min()
imb_max = all_imbalances.max()
imb_margin = (imb_max - imb_min) * 0.2
imbalance_ylim = [imb_min - imb_margin, imb_max + imb_margin]

# Prepare data for plotting - 2 subplots per supplier
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Plot for each supplier
for idx, supplier in enumerate(['SUP_A', 'SUP_B']):
    # Filter data for this supplier
    supplier_data = es_monthly_df_analysis[es_monthly_df_analysis['supplier_id'] == supplier].copy()
    
    # Get supplier display name
    supplier_display = f"{supplier} ({supplier_names[supplier]})"
    
    # Sort by datetime for proper line plot
    supplier_data = supplier_data.sort_values('datetime')
    
    # Create x-axis positions
    x_pos = range(len(supplier_data))
    
    # Plot 1: Financial overview (revenue, costs, profit/loss) - SAME SCALE
    ax1 = axes[idx, 0]
    ax1.plot(x_pos, supplier_data['total_revenue_eur'], marker='o', linewidth=2.5, 
            label='Total Revenue', color='green', markersize=8)
    ax1.plot(x_pos, supplier_data['total_costs_eur'], marker='s', linewidth=2.5, 
            label='Total Costs', color='red', markersize=8)
    ax1.plot(x_pos, supplier_data['profit_loss_eur'], marker='^', linewidth=2.5, 
            label='Profit/Loss', color='blue', markersize=8, linestyle='--')
    
    # Add horizontal line at zero
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
    
    # Formatting for plot 1 with SAME Y-AXIS
    ax1.set_xlabel('Month', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Amount (EUR)', fontsize=12, fontweight='bold')
    ax1.set_title(f'{supplier_display} - Financial Overview (with REC)', fontsize=14, fontweight='bold')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(supplier_data['datetime'].values, rotation=45, ha='right')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
    ax1.set_ylim(financial_ylim)  # SAME SCALE FOR BOTH
    
    # Plot 2: Balancing Group Position & Imbalance - SAME SCALE
    ax2 = axes[idx, 1]
    
    # Plot all three values on same scale for direct comparison with transparency
    ax2.plot(x_pos, supplier_data['imbalance_mwh'], marker='^', linewidth=2.5, 
            label='Imbalance (Actual-Forecast)', color='#3498db', markersize=8, linestyle='--', alpha=0.9)
    ax2.plot(x_pos, supplier_data['balancing_group_forecast_mwh'], marker='s', linewidth=2.5, 
            label='BG Forecast (ID)', color='#2ecc71', markersize=8, alpha=0.6)
    ax2.plot(x_pos, supplier_data['balancing_group_actual_mwh'], marker='o', linewidth=2.5, 
            label='BG Actual (Load-Gen)', color='#e74c3c', markersize=8, alpha=0.9)
    
    # Add horizontal line at zero
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.7)
    
    # Formatting for plot 2 with SAME Y-AXIS
    ax2.set_xlabel('Month', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Position / Imbalance (MWh)', fontsize=12, fontweight='bold')
    ax2.set_title(f'{supplier_display} - Balancing Group Position & Imbalance (with REC)', fontsize=14, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(supplier_data['datetime'].values, rotation=45, ha='right')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.3f}'))
    ax2.set_ylim(imbalance_ylim)  # SAME SCALE FOR BOTH

plt.tight_layout()
plt.show()

# Stacked bar chart for revenue and cost components with SAME SCALES
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Calculate global max for revenue and cost scales
max_revenue_stack = all_data.groupby(['datetime', 'supplier_id']).agg({
    'revenue_energy_market_sales_eur': 'sum',
    'revenue_balancing_rewards_eur': 'sum',
    'revenue_retail_sales_eur': 'sum'
}).sum(axis=1).max()

max_cost_stack = all_data.groupby(['datetime', 'supplier_id']).agg({
    'cost_energy_market_purchases_eur': 'sum',
    'cost_balancing_penalties_eur': 'sum',
    'cost_retail_purchases_eur': 'sum'
}).sum(axis=1).max()

revenue_ylim = [0, max_revenue_stack * 1.1]
cost_ylim = [0, max_cost_stack * 1.1]

for idx, supplier in enumerate(['SUP_A', 'SUP_B']):
    supplier_data = es_monthly_df_analysis[es_monthly_df_analysis['supplier_id'] == supplier].copy()
    supplier_data = supplier_data.sort_values('datetime')
    n_months = len(supplier_data)
    
    # Get supplier display name
    supplier_display = f"{supplier} ({supplier_names[supplier]})"
    
    # Revenue breakdown with SIDE-BY-SIDE bars
    ax_rev = axes[idx, 0]
    revenue_components = [
        supplier_data['revenue_energy_market_sales_eur'].values,
        supplier_data['revenue_balancing_rewards_eur'].values,
        supplier_data['revenue_retail_sales_eur'].values
    ]
    labels_rev = ['Energy Market Sales', 'Balancing Rewards', 'Retail Sales']
    colors_rev = ['#27ae60', '#f39c12', '#3498db']  # Dark green, orange, blue
    
    # Bar width and positions for grouped bars
    bar_width = 0.25
    x_pos = np.arange(n_months)
    
    for i, (component, label, color) in enumerate(zip(revenue_components, labels_rev, colors_rev)):
        offset = (i - 1) * bar_width  # Center the groups
        ax_rev.bar(x_pos + offset, component, width=bar_width, label=label, color=color, alpha=0.8)
    
    ax_rev.set_xlabel('Month', fontsize=11, fontweight='bold')
    ax_rev.set_ylabel('Revenue (EUR)', fontsize=11, fontweight='bold')
    ax_rev.set_title(f'{supplier_display} - Revenue Components', fontsize=13, fontweight='bold')
    ax_rev.set_xticks(x_pos)
    ax_rev.set_xticklabels(supplier_data['datetime'].values, rotation=45, ha='right')
    ax_rev.legend(loc='upper left', fontsize=9)
    ax_rev.grid(True, alpha=0.3, axis='y')
    ax_rev.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
    ax_rev.set_ylim(revenue_ylim)  # SAME SCALE FOR BOTH
    
    # Cost breakdown with SAME SCALE
    # Cost breakdown with SIDE-BY-SIDE bars
    ax_cost = axes[idx, 1]
    cost_components = [
        supplier_data['cost_energy_market_purchases_eur'].values,
        supplier_data['cost_balancing_penalties_eur'].values,
        supplier_data['cost_retail_purchases_eur'].values
    ]
    labels_cost = ['Energy Market Purchases', 'Balancing Penalties', 'Retail Purchases']
    colors_cost = ['#e74c3c', '#8e44ad', '#e67e22']  # Red, purple, orange
    
    for i, (component, label, color) in enumerate(zip(cost_components, labels_cost, colors_cost)):
        offset = (i - 1) * bar_width  # Center the groups
        ax_cost.bar(x_pos + offset, component, width=bar_width, label=label, color=color, alpha=0.8)
    
    ax_cost.set_xlabel('Month', fontsize=11, fontweight='bold')
    ax_cost.set_ylabel('Costs (EUR)', fontsize=11, fontweight='bold')
    ax_cost.set_title(f'{supplier_display} - Cost Components', fontsize=13, fontweight='bold')
    ax_cost.set_xticks(x_pos)
    ax_cost.set_xticklabels(supplier_data['datetime'].values, rotation=45, ha='right')
    ax_cost.legend(loc='upper left', fontsize=9)
    ax_cost.grid(True, alpha=0.3, axis='y')
    ax_cost.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
    ax_cost.set_ylim(cost_ylim)  # SAME SCALE FOR BOTH

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*80)
print("ANNUAL FINANCIAL SUMMARY - WITH REC")
print("="*80)
# Print summary statistics
for supplier in ['SUP_A', 'SUP_B']:
    supplier_data = es_monthly_df_analysis[es_monthly_df_analysis['supplier_id'] == supplier].copy()
    supplier_display = f"{supplier} ({supplier_names[supplier]})"
    
    print(f"\n{supplier_display}:")
    print(f"\n  REVENUES:")
    print(f"    Energy Market Sales:    €{supplier_data['revenue_energy_market_sales_eur'].sum():,.2f}")
    print(f"    Balancing Rewards:      €{supplier_data['revenue_balancing_rewards_eur'].sum():,.2f}")
    print(f"    Retail Sales:           €{supplier_data['revenue_retail_sales_eur'].sum():,.2f}")
    print(f"    ──────────────────────────────────────")
    print(f"    Total Revenue:          €{supplier_data['total_revenue_eur'].sum():,.2f}")
    
    print(f"\n  COSTS:")
    print(f"    Energy Market Purchases: €{supplier_data['cost_energy_market_purchases_eur'].sum():,.2f}")
    print(f"    Balancing Penalties:     €{supplier_data['cost_balancing_penalties_eur'].sum():,.2f}")
    print(f"    Retail Purchases:        €{supplier_data['cost_retail_purchases_eur'].sum():,.2f}")
    print(f"    ──────────────────────────────────────")
    print(f"    Total Costs:             €{supplier_data['total_costs_eur'].sum():,.2f}")
    
    print(f"\n  PROFIT/LOSS:")
    print(f"    Annual Total:            €{supplier_data['profit_loss_eur'].sum():,.2f}")
    print(f"    Monthly Average:         €{supplier_data['profit_loss_eur'].mean():,.2f}")
    print(f"    Monthly Min:             €{supplier_data['profit_loss_eur'].min():,.2f}")
    print(f"    Monthly Max:             €{supplier_data['profit_loss_eur'].max():,.2f}")
    
    # Imbalance statistics
    print(f"\n  IMBALANCE STATISTICS:")
    
    # Determine system position based on imbalance
    total_imb = supplier_data['imbalance_mwh'].sum()
    if total_imb > 0:
        position = "SHORT (Deficit)"
        position_desc = "Actual deficit > forecast, need to buy more"
    elif total_imb < 0:
        position = "LONG (Surplus)"
        position_desc = "Actual deficit < forecast, have surplus to sell"
    else:
        position = "BALANCED"
        position_desc = "Actual matched forecast exactly"
    
    print(f"    System Position:         {position}")
    print(f"                             ({position_desc})")
    print(f"    Actual BG Position:      {supplier_data['balancing_group_actual_mwh'].sum():,.2f} MWh")
    print(f"    Avg Monthly Actual:      {supplier_data['balancing_group_actual_mwh'].mean():,.2f} MWh")
    print(f"    Max Monthly Actual:      {abs(supplier_data['balancing_group_actual_mwh']).max():,.2f} MWh")
    print(f"    Forecast BG Position:    {supplier_data['balancing_group_forecast_mwh'].sum():,.2f} MWh")
    print(f"    Avg Monthly Forecast:    {supplier_data['balancing_group_forecast_mwh'].mean():,.2f} MWh")
    print(f"    Max Monthly Forecast:    {abs(supplier_data['balancing_group_forecast_mwh']).max():,.2f} MWh")
    print(f"    Total Imbalance:         {supplier_data['imbalance_mwh'].sum():,.2f} MWh")
    print(f"    Avg Monthly Imbalance:   {supplier_data['imbalance_mwh'].mean():,.2f} MWh")
    print(f"    Max Monthly Imbalance:   {abs(supplier_data['imbalance_mwh']).max():,.2f} MWh")
    
    # REC statistics
    if 'internal_shared_energy_mwh' in supplier_data.columns:
        print(f"\n  REC INTERNAL SHARING:")
        print(f"    Total Energy Shared:     {supplier_data['internal_shared_energy_mwh'].sum():,.2f} MWh")
        print(f"    Monthly Average:         {supplier_data['internal_shared_energy_mwh'].mean():,.2f} MWh")
        print(f"    Monthly Max:             {supplier_data['internal_shared_energy_mwh'].max():,.2f} MWh")

print("\n" + "="*80)

In [ ]:
# Debug: Check available columns in monthly dataframe
print("Available columns in es_monthly_df:")
print(es_monthly_df.columns.tolist())
print(f"\nDataframe shape: {es_monthly_df.shape}")

# Check if balancing columns exist in timeseries
print("\nAvailable columns in es_timeseries_df:")
print([col for col in es_timeseries_df.columns if 'imbalance' in col.lower() or 'penalty' in col.lower() or 'reward' in col.lower()])

## Summary: REC Settlement Impact on Financial Calculations

### ✅ CONFIRMED: All Downstream Calculations Use Corrected Values

The B2 notebook (with REC) correctly implements the following data flow:

#### **1. Original Data → REC Settlement → Corrected Data**

```python
# INPUT: Original meter readings
es_data['load_actual']  # Original load from SimBench
es_data['res_actual']   # Original generation from SimBench

# REC SETTLEMENT: Internal energy sharing
es_timeseries_df, corrected_load_df, corrected_gen_df = calculate_rec_settlement(
    config, 
    es_timeseries_df,
    es_data['load_actual'],  # Original
    es_data['res_actual']    # Original
)

# OUTPUT: Corrected meter readings (after internal sharing)
corrected_load_df   # Reduced load (after REC sharing)
corrected_gen_df    # Reduced generation (after REC sharing)
```

#### **2. Balancing Market Uses Corrected Values**

```python
es_timeseries_df = calculate_balancing_market_positions(
    es_timeseries_df,
    corrected_load_df,  # ✅ Uses corrected load
    corrected_gen_df,   # ✅ Uses corrected generation
    es_data['prices'],
    config
)

# Creates columns with corrected values:
# - actual_load_mwh (corrected)
# - actual_gen_mwh (corrected)
# - load_imbalance_mwh (calculated from corrected values)
# - gen_imbalance_mwh (calculated from corrected values)
# - imbalance_penalty (based on corrected imbalances)
# - imbalance_reward (based on corrected imbalances)
```

#### **3. Customer Billing Uses Corrected Values**

```python
customer_billing_df = calculate_customer_bills(
    config,
    corrected_load_df,  # ✅ Uses corrected load
    corrected_gen_df,   # ✅ Uses corrected generation
    es_data['prices']
)

# Creates billing based on corrected grid exchange:
# - sales_revenue_eur (supplier → customer, reduced by REC sharing)
# - purchase_costs_eur (customer → supplier, reduced by REC sharing)
```

#### **4. Financial Analysis Uses Aggregated Corrected Values**

```python
# Monthly aggregation sums corrected values:
es_monthly_df_analysis['revenue_retail_sales_eur'] = Σ(sales_revenue_eur)
es_monthly_df_analysis['cost_retail_purchases_eur'] = Σ(purchase_costs_eur)
es_monthly_df_analysis['cost_balancing_penalties_eur'] = Σ(imbalance_penalty)
es_monthly_df_analysis['revenue_balancing_rewards_eur'] = Σ(imbalance_reward)

# All based on CORRECTED grid exchange (after REC internal settlement)
```

---

### **Financial Impact of Using Corrected Values**

| Metric | Without REC (Original) | With REC (Corrected) | Impact |
|--------|----------------------|---------------------|---------|
| Grid Export (Prosumers) | Higher | Lower | ↓ Supplier feed-in costs |
| Grid Import (All Members) | Higher | Lower | ↓ Supplier revenue from retail |
| Balancing Imbalances | Higher | Lower | ↓ Balancing penalties |
| Wholesale Purchases | Higher | Lower | ↓ Market purchase costs |

**Key Benefit:** REC internal sharing reduces external grid dependency, lowering costs for suppliers and improving system efficiency.

---

### **Terminology Clarification**

⚠️ **Important:** In B2 results, the term "actual" means "actual grid exchange" (corrected values), NOT original meter readings.

- `actual_load_mwh` = Corrected load after REC sharing
- `actual_gen_mwh` = Corrected generation after REC sharing
- Original meter readings are stored in `es_data['load_actual']` and `es_data['res_actual']`
- Corrected values are stored in `corrected_load_df` and `corrected_gen_df`

This ensures all financial calculations reflect the **true grid exchange** after community self-consumption.